In [259]:
!pip install torch-geometric
!pip install python-sat[pblib,aiger]

!rm -r ./NNSAT_Project/Cache
!rm -r ./NNSAT_Project/Checkpoints
!rm -r ./NNSAT_Project/Logs

In [260]:
import os
import re
import time
import random

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.decomposition import TruncatedSVD, PCA

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, ConcatDataset

from torch_geometric.data import HeteroData
import torch_geometric.nn as gnn
from torch_geometric.utils import scatter
from torch_geometric.loader import DataLoader

import optuna
from optuna.pruners import SuccessiveHalvingPruner
import matplotlib.pyplot as plt



# ---------------------------------------------------------
# Configuration & Setup
# ---------------------------------------------------------
# DATA_PATH = '/kaggle/input/datasets/weihengwong/dataset' 
# DATA_PATH = '/kaggle/input/datasets/weihengwong/test-dataset/test_dataset'
DATA_PATH = '/kaggle/input/datasets/weihengwong/general-problems'
# DATA_PATH = '/kaggle/input/datasets/weihengwong/hard-problems'
CACHE_PATH = './NNSAT_Project/Cache' 
CHECKPOINT_PATH = './NNSAT_Project/Checkpoints' 
LOG_PATH = './NNSAT_Project/Logs'
LOAD_PATH = '/kaggle/input/datasets/weihengwong/perfect-refine/'
# LOAD_PATH = '/kaggle/input/datasets/weihengwong/ucloss-refine'
# LOAD_PATH = '/kaggle/input/datasets/weihengwong/together-tuned'

LOAD_PATH_CORE = '/kaggle/input/datasets/weihengwong/best-sru140-model/M-Trial4-T26-D64-L3.27e-05_epoch127_BEST.pth'
LOAD_PATH_CTP = '/kaggle/input/datasets/weihengwong/ctp-mlp-model/ClauseTierPredictor-D220-Dr0.30486349258950407-L0.0010006590443426412_best.pth'
LOAD_PATH_ADM = '/kaggle/input/datasets/weihengwong/adm-mlp-model/AssignmentDecodingModel-D207-Dr0.11160159495261959-L0.0008748982734118006_best.pth'


os.makedirs(CACHE_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(LOG_PATH, exist_ok=True)

BATCH_SIZE = 128

In [261]:
# walksat

def walksat(clauses, n_vars, max_flips=10000, p_noise=0.5, initial_assignment=None):
    if initial_assignment is not None:
        assignment = np.array(initial_assignment, dtype=int)
    else:
        assignment = np.random.randint(2, size=n_vars)

    # Pre-process clauses for faster access
    c_vars = [[abs(l)-1 for l in c] for c in clauses] # list of lists of variable indices 
    c_pols = [[1 if l > 0 else 0 for l in c] for c in clauses] # list of lists of polarities (1 if pos, 0 if neg)
    
    # Build a map of {variable_index -> [clause_indices]}
    # This allows us to only update relevant clauses when we flip a variable
    var_to_clauses = [[] for _ in range(n_vars)]
    for i, c in enumerate(c_vars):
        for v in c:
            var_to_clauses[v].append(i)

    # Compute Initial Satisfaction
    sat_counts = np.zeros(len(clauses), dtype=int)
    unsat_stack = [] 
    for i, (lits, pols) in enumerate(zip(c_vars, c_pols)):
        satisfied_term_count = 0
        for var_idx, pol in zip(lits, pols):
            if assignment[var_idx] == pol:
                satisfied_term_count += 1
        
        sat_counts[i] = satisfied_term_count
        if satisfied_term_count == 0:
            unsat_stack.append(i)

    for step in range(max_flips):
        if len(unsat_stack) == 0:
            return assignment, step # solved
            
        rand_idx = random.randint(0, len(unsat_stack) - 1)
        target_c_idx = unsat_stack[rand_idx]
        
        candidates = c_vars[target_c_idx]

        best_break = float('inf')
        best_candidates = []
        
        for candidate in candidates:
            break_score = 0
            
            # Check all clauses this variable appears in
            for c_idx in var_to_clauses[candidate]:
                if sat_counts[c_idx] == 1:
                    # Verify if candidate is the one satisfying it
                    c_lits = c_vars[c_idx]
                    c_ps = c_pols[c_idx]
                    idx_in_c = c_lits.index(candidate)
                    
                    if assignment[candidate] == c_ps[idx_in_c]:
                        break_score += 1
            
            if break_score < best_break:
                best_break = break_score
                best_candidates = [candidate]
            elif break_score == best_break:
                best_candidates.append(candidate)

        # Free Move
        if best_break == 0:
            flip_var = random.choice(best_candidates)
        # Random Move
        elif random.random() < p_noise:
            flip_var = random.choice(candidates)
        # Greedy Move
        else:            
            flip_var = random.choice(best_candidates)

        # apply flip
        old_val = assignment[flip_var]
        new_val = 1 - old_val
        assignment[flip_var] = new_val
        
        # Update SAT Counts
        # We only check clauses that contain the flipped variable
        for clause_idx in var_to_clauses[flip_var]:
            # Get the polarity of the flipped variable in this specific clause
            lits = c_vars[clause_idx]
            pols = c_pols[clause_idx]
            idx = lits.index(flip_var)
            pol = pols[idx]
            
            # If new_val matches polarity -> We just satisfied it -> count +1
            # If old_val matched polarity -> We just broke it -> count -1
            if new_val == pol:
                sat_counts[clause_idx] += 1
            else:
                sat_counts[clause_idx] -= 1
        
        # refresh UNSAT Stack
        unsat_stack = np.where(sat_counts == 0)[0]

    return None, max_flips # Failed


def get_close_assignment(assignment, clauses, n_vars):
    sol, flips = walksat(clauses, n_vars=n_vars, initial_assignment=assignment, max_flips=200000)
    # if sol is None:
    #     raise Exception("Solution not found")
    # else:
    #     return sol
    return sol


from pysat.solvers import Minisat22
def get_truth_assignment(clauses):
    """Finds a single satisfying assignment using Minisat."""
    with Minisat22(bootstrap_with=clauses) as solver:
        if solver.solve():
            literals = set(solver.get_model()) # type: ignore
            # Convert PySAT model to a binary [0, 1] array
            return [1 if (i + 1) in literals else 0 for i in range(len(literals))]
        return None


def clause_satlit_count(clauses, assignment):
    lit_sat = []
    for c in clauses:
        n_sat = 0
        for j, k in enumerate(assignment):
            lit = j + 1
            if k == 1 and lit in c:
                n_sat += 1
            elif k == 0 and -lit in c:
                n_sat += 1
        lit_sat.append(n_sat)
    return lit_sat

In [262]:
# Data processing
# ---------------------------------------------------------
# Data Processing Functions
# ---------------------------------------------------------

def dimacs_parser(filepath):
    """Parses a DIMACS .cnf file to extract variables, clauses, and structure."""
    try:
        with open(filepath, 'r', encoding="utf-8") as file:
            lines = file.read().split('\n')
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return 0, 0, []

    n_var = 0
    n_clause = 0
    clauses = []
    
    for line in lines:
        if not line or line[0] == "c":
            continue
            
        info = [i for i in line.split(' ') if i]

        if info[0] == "p":
            n_var = int(info[2])
            n_clause = int(info[3])
        elif line[-1] == "0" and len(line) > 1:
            clause = [int(i) for i in info[:-1]]
            clauses.append(clause)
            
    return n_var, n_clause, clauses


def read_data(folder_name, is_training=True, fixed_label=None, limit=float('inf'), generate_labels=False):
    """Unified function to read both training and testing datasets."""
    data_set = []  
    folder_path = os.path.join(DATA_PATH, folder_name)
    filenames = sorted(os.listdir(folder_path))
    count = 0
    
    for filename in filenames:
        if not filename.endswith('.cnf'):
            continue

        filepath = os.path.join(folder_path, filename)
        n_var, n_clauses, clauses = dimacs_parser(filepath)

        # 1. Determine Satisfiability (Label)
        if is_training:
            if '_SAT.cnf' in filename:
                label = 1
            elif '_UNSAT.cnf' in filename:
                label = 0
            else:
                continue # Skip unknown formats
        else:
            label = fixed_label # Use provided label for test sets
            
        is_sat_ground_truth = (label == 1)

        # 2. Process based on generate_labels flag
        if not generate_labels:
            # Standard Processing
            data_set.append((clauses, n_var, label))
            
        else:
            # Supervised Processing
            if is_sat_ground_truth:            
                assignment = np.array(get_truth_assignment(clauses))
                flipped = 1 - assignment
                full_assignment = np.concatenate([assignment, flipped], axis=0) 
                lit_labels = full_assignment
                labels = np.array(clause_satlit_count(clauses, assignment))
                clause_labels = np.where(labels == 1, 0, np.where(labels == 2, 1, 2))
                
                data_set.append({
                    'clauses': clauses,
                    'n_var': n_var,
                    'is_sat': 1,
                    'lit_labels': torch.tensor(lit_labels, dtype=torch.long),
                    'clause_labels': torch.tensor(clause_labels, dtype=torch.long)
                })
            else:
                # Dummy labels for UNSAT to maintain batch shapes
                data_set.append({
                    'clauses': clauses,
                    'n_var': n_var,
                    'is_sat': 0,
                    'lit_labels': torch.zeros(n_var * 2, dtype=torch.long),
                    'clause_labels': torch.zeros(n_clauses, dtype=torch.long)
                })

        count += 1
        if count == limit:
            break
            
    return data_set


def build_sparse_edges(clauses, n_var):
    """
    Directly builds PyTorch sparse edge indices.
    Bypasses the memory limit of creating a dense matrix first.
    """
    source_nodes = [] # Literals
    target_nodes = [] # Clauses

    for clause_idx, clause in enumerate(clauses):
        for literal in clause:
            if literal > 0:
                # Positive literals map to indices 0 to (n_var - 1)
                lit_idx = literal - 1
            else:
                # Negative literals map to indices n_var to (2*n_var - 1)
                lit_idx = n_var + abs(literal) - 1
            
            source_nodes.append(lit_idx)
            target_nodes.append(clause_idx)
            
    # Create [2, num_edges] tensor
    edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
    edge_attr = torch.ones(len(source_nodes), dtype=torch.float32)

    return edge_index, edge_attr

# ---------------------------------------------------------
# Unified Dataset Class
# ---------------------------------------------------------

class SATDataset(Dataset): 
    """
    A unified dataset class that handles both training and testing data,
    processing graphs into HeteroData objects and caching them to disk.
    """
    def __init__(self, data_file, is_training=True, fixed_label=None, limit=float('inf'), generate_labels=False):
        super().__init__()
        self.data_file = data_file
        self.is_training = is_training
        self.generate_labels = generate_labels
        
        # Name the cache file uniquely based on the data type
        prefix = "train" if is_training else f"test_{fixed_label}"
        label_tag = "_labels" if generate_labels else ""
        self.cache_file_path = os.path.join(CACHE_PATH, f"cached_{prefix}_{data_file}{label_tag}.pt")
        
        if os.path.exists(self.cache_file_path):
            print(f"Loading cached data from {self.cache_file_path}...")
            self.data_list = torch.load(self.cache_file_path, weights_only=False)
        else:
            print(f"Cache file not found. Reading and converting raw data...")
            raw_problems = read_data(data_file, is_training=is_training, fixed_label=fixed_label, limit=limit, generate_labels=generate_labels)
            if not generate_labels:   
                self.data_list = self._process(raw_problems)
            else:
                self.data_list = self._process_labels(raw_problems)
                

        print(f"Loaded {len(self.data_list)} samples total.")

    def _process(self, raw_problems):
        processed_list = []

        for clauses, n_var, label in raw_problems:
            n_clauses = len(clauses)
            n_literals = n_var * 2
    
            # Get edges directly (no dense matrix memory overhead)
            edge_index, edge_attr = build_sparse_edges(clauses, n_var)
    
            # Build PyTorch Geometric Heterogeneous graph
            data = HeteroData()
            data['literal'].num_nodes = n_literals
            data['clause'].num_nodes = n_clauses
            data['literal', 'to', 'clause'].edge_index = edge_index
            data['literal', 'to', 'clause'].edge_attr = edge_attr
    
            data.y = torch.tensor([label], dtype=torch.float)
            data.n_vars = torch.tensor([n_var], dtype=torch.long)
            data.n_clauses = torch.tensor([n_clauses], dtype=torch.long)

            processed_list.append(data)

        print(f"Saving processed data to {self.cache_file_path}...")
        torch.save(processed_list, self.cache_file_path)

        return processed_list

    def _process_labels(self, raw_problems):
        processed_list = []

        for instance in raw_problems:
            clauses = instance['clauses']
            n_var = instance['n_var']
            label = instance['is_sat']
            lit_label = instance['lit_labels']
            clause_label = instance['clause_labels']

            n_clauses = len(clauses)
            n_literals = n_var * 2
    
            # Get edges directly (no dense matrix memory overhead)
            edge_index, edge_attr = build_sparse_edges(clauses, n_var)
    
            # Build PyTorch Geometric Heterogeneous graph
            data = HeteroData()
            data['literal'].num_nodes = n_literals
            data['clause'].num_nodes = n_clauses
            data['literal', 'to', 'clause'].edge_index = edge_index
            data['literal', 'to', 'clause'].edge_attr = edge_attr
    
            data.y = torch.tensor([label], dtype=torch.float)
            data.n_vars = torch.tensor([n_var], dtype=torch.long)
            data.n_clauses = torch.tensor([n_clauses], dtype=torch.long)
            data.lit_label = lit_label
            data.clause_label = clause_label

            processed_list.append(data)

        print(f"Saving processed data to {self.cache_file_path}...")
        torch.save(processed_list, self.cache_file_path)

        return processed_list
            
    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]


def training_data_setup(dataset, generate_labels=False):
    train_data = SATDataset(data_file=dataset, is_training=True, generate_labels=generate_labels)
    return DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)


def testing_data_setup(sat, unsat, generate_labels=False):
    test_sat_data = SATDataset(data_file=sat, is_training=False, fixed_label=1, generate_labels=generate_labels)
    test_unsat_data = SATDataset(data_file=unsat, is_training=False, fixed_label=0, generate_labels=generate_labels)
    
    merged_test_data = ConcatDataset([test_sat_data, test_unsat_data])
    return DataLoader(merged_test_data, batch_size=BATCH_SIZE, shuffle=False)

In [263]:
# NeuroSAT class
# ==============================================================================
# 1. Helper Modules 
# ==============================================================================

class MLP(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.ReLU(),
            nn.Linear(hid_dim, out_dim)
        )
        
    def forward(self, x):
        return self.net(x)


def flip_index(L, n_literals_list):
    device = L.device
    
    # 1. Get the number of variables (V) for each graph in the batch
    n_vars = n_literals_list // 2
    
    # 2. Create the shift values: [V1, -V1, V2, -V2, ...]
    shift_vals = torch.stack([n_vars, -n_vars], dim=1).view(-1)
    
    # 3. Create the repeat counts: [V1, V1, V2, V2, ...]
    repeat_vals = torch.stack([n_vars, n_vars], dim=1).view(-1)
    
    # 4. Interleave to get the exact shift for every single literal
    shifts = torch.repeat_interleave(shift_vals, repeat_vals)
    
    # 5. Add the shifts to a standard arange index
    flip_idx = torch.arange(L.size(0), device=device) + shifts
    
    return flip_idx

# ==============================================================================
# 2. The Core Network
# ==============================================================================

class NeuroSATNetwork(nn.Module):
    def __init__(self, d_model=64, T=26):
        super().__init__()
        self.d = d_model
        self.T = T

        self.L_init = nn.Parameter(torch.randn(1, d_model))
        self.C_init = nn.Parameter(torch.randn(1, d_model))

        self.L_msg = MLP(d_model, d_model, d_model)
        self.C_msg = MLP(d_model, d_model, d_model)

        self.L_update = nn.LSTMCell(3 * d_model, d_model)
        self.C_update = nn.LSTMCell(2 * d_model, d_model)

        self.L_norm = nn.LayerNorm(d_model)
        self.C_norm = nn.LayerNorm(d_model)
        
        self.L_vote = MLP(d_model, d_model, 1)

    def forward(self, data):
        edge_index = data['literal', 'to', 'clause'].edge_index
        n_literals = data['literal'].num_nodes 
        n_clauses = data['clause'].num_nodes 
        n_graphs = data.num_graphs
        
        n_literals_list = torch.bincount(data['literal'].batch)      

        L_h = self.L_init.repeat(n_literals, 1) / (self.d ** 0.5)
        C_h = self.C_init.repeat(n_clauses, 1) / (self.d ** 0.5)
        L_c = torch.zeros_like(L_h)
        C_c = torch.zeros_like(C_h)

        # PRE-COMPUTE FLIP INDEX ONCE
        flip_idx = flip_index(L_h, n_literals_list)

        edge_index = edge_index.to(L_h.device)

        for t in range(self.T):
            L_pre = self.L_msg(L_h)  
            L_msgs = L_pre[edge_index[0]]  
            C_aggr = scatter(L_msgs, edge_index[1], dim=0, dim_size=n_clauses, reduce='sum') 
            
            C_input = torch.cat([C_h, C_aggr], dim=1)
            C_h, C_c = self.C_update(C_input, (C_h, C_c))
            C_h = self.C_norm(C_h)

            C_pre = self.C_msg(C_h)
            C_msgs = C_pre[edge_index[1]]
            L_aggr = scatter(C_msgs, edge_index[0], dim=0, dim_size=n_literals, reduce='sum')
            
            L_flipped = L_h[flip_idx]
            L_input = torch.cat([L_h, L_flipped, L_aggr], dim=1)
            L_h, L_c = self.L_update(L_input, (L_h, L_c))
            L_h = self.L_norm(L_h)

        L_votes = self.L_vote(L_h).squeeze(-1) 
        batch_idx = data['literal'].batch
        graph_logits = scatter(L_votes, batch_idx, dim=0, dim_size=n_graphs, reduce='mean')

        L_votes_flipped = L_votes[flip_idx]
        averaged_literal_votes = (L_votes + L_votes_flipped) / 2
        n_vars_tensor = n_literals_list // 2
        bool_pattern = torch.tensor([True, False], device=L_h.device).repeat(n_graphs)
        repeat_counts = torch.stack([n_vars_tensor, n_vars_tensor], dim=1).view(-1)
        pos_mask = torch.repeat_interleave(bool_pattern, repeat_counts)
        
        var_votes = averaged_literal_votes[pos_mask]

        # Return everything: Graph Votes, Literal Embeddings, Clause Embeddings, Variable Votes
        return graph_logits, L_h, C_h, var_votes
            

# ==============================================================================
# 3. The MLP heads
# ==============================================================================

# ADM + CTP
class MlpHeadModel(nn.Module):
    def __init__(self, in_dim=65, hid_dim=128, out_dim=2):
        super(MlpHeadModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.BatchNorm1d(hid_dim), 
            nn.ReLU(),
            
            nn.Linear(hid_dim, hid_dim // 2),
            nn.BatchNorm1d(hid_dim // 2),
            nn.ReLU(),
            
            nn.Linear(hid_dim // 2, out_dim)
        )
        
    def forward(self, x):
        return self.network(x)



class SepMlpHeadModel(nn.Module):
    def __init__(self, in_dim=65, hid_dim=128, out_dim=2):
        super(SepMlpHeadModel, self).__init__()
        # 64 embeddings + 1 Vote = 65 inputs
        self.norm = nn.LayerNorm(65)
        
        self.network = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.ReLU(),
            nn.Dropout(0.2), # Prevents overfitting
            nn.Linear(hid_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim) # Output: [False, True]
        )
        
    def forward(self, x):
        x = self.norm(x)
        return self.network(x)
        

In [264]:
# MTL

import torch
import torch.nn as nn

class MtlNeuroSAT(nn.Module):
    def __init__(self, d_model=64, T=26):
        super(MtlNeuroSAT, self).__init__()
        self.core = NeuroSATNetwork(d_model=d_model, T=T)
        self.adm = MlpHeadModel(in_dim=d_model+1, hid_dim=207, out_dim=2)
        self.ctp = MlpHeadModel(in_dim=d_model+1, hid_dim=220, out_dim=3)
        
    def forward(self, batch_data):
        """Unified forward pass for all three networks."""
        outputs, L_h, C_h, var_votes = self.core(batch_data)
        adm_logits = self.adm(L_h)
        ctp_logits = self.ctp(C_h)
        
        return outputs, adm_logits, ctp_logits


class SepMtlNeuroSAT(nn.Module):
    def __init__(self, d_model=64, T=26):
        super(SepMtlNeuroSAT, self).__init__()
        self.core = NeuroSATNetwork(d_model=d_model, T=T)
        self.adm = SepMlpHeadModel(in_dim=d_model+1, hid_dim=207, out_dim=2)
        self.ctp = SepMlpHeadModel(in_dim=d_model+1, hid_dim=220, out_dim=3)
        
    def forward(self, batch_data):
        """Unified forward pass for all three networks."""
        outputs, L_h, C_h, var_votes = self.core(batch_data)
        adm_logits = self.adm(L_h)
        ctp_logits = self.ctp(C_h)
        
        return outputs, adm_logits, ctp_logits


# uncertainty weights
class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, n_tasks=3):
        super().__init__()
        # log(sigma^2) for each task - learnable
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))
    
    def forward(self, losses):
        total = 0
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total += precision * loss + 0.5 * self.log_vars[i]
        return total

In [265]:
# MTL full

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.utils import scatter

class MtlTrainer:
    """
    Manages the 3-Stage Training Pipeline, Checkpointing, and Evaluation 
    for the MtlNeuroSAT architecture.
    """
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        
        # Loss Functions
        self.global_loss_fn = nn.BCEWithLogitsLoss()
        self.mlp_loss_fn = nn.CrossEntropyLoss(reduction='none')
        self.uc_loss_fn = UncertaintyWeightedLoss(n_tasks=3).to(device)

        # State
        self.optimizer = None

    # =========================================================================
    # Optimizer & Checkpoint Management
    # =========================================================================

    def update_stage_optimizer(self, stage, lrs=None):
        """Dynamically freezes layers and adjusts learning rates."""
        if lrs is None:
            lrs = {}
        lr_core = lrs.get('core', 5e-5)
        lr_adm  = lrs.get('adm',  5e-5)
        lr_ctp  = lrs.get('ctp',  5e-5)
        lr_uc   = lrs.get('uc',   5e-5)

        if stage == 1:
            for p in self.model.core.parameters(): p.requires_grad = True
            for p in self.model.adm.parameters(): p.requires_grad = False 
            for p in self.model.ctp.parameters(): p.requires_grad = False 
            for p in self.uc_loss_fn.parameters(): p.requires_grad = False 
            self.optimizer = optim.Adam(self.model.core.parameters(), lr=lr_core, weight_decay=1e-10)
            
        elif stage == 2:
            for p in self.model.core.parameters(): p.requires_grad = False
            for p in self.model.adm.parameters(): p.requires_grad = True
            for p in self.model.ctp.parameters(): p.requires_grad = True
            for p in self.uc_loss_fn.parameters(): p.requires_grad = False
            self.optimizer = optim.Adam([
                {'params': self.model.adm.parameters(), 'lr': lr_adm},
                {'params': self.model.ctp.parameters(), 'lr': lr_ctp},
            ], weight_decay=1e-10)
                
        elif stage == 3:
            for p in self.model.core.parameters(): p.requires_grad = True
            for p in self.uc_loss_fn.parameters(): p.requires_grad = True
            self.optimizer = optim.Adam([
                {'params': self.model.core.parameters(), 'lr': lr_core}, 
                {'params': self.model.adm.parameters(), 'lr': lr_adm},
                {'params': self.model.ctp.parameters(), 'lr': lr_ctp},
                {'params': self.uc_loss_fn.parameters(), 'lr': lr_uc}
            ], weight_decay=1e-10)

        elif stage == 4:
            for p in self.model.parameters(): p.requires_grad = True
            self.optimizer = optim.Adam(self.model.parameters(), lr=1e-5, weight_decay=1e-10)
            
    def save(self, epoch, loss, filename):
        checkpoint = {
            'epoch': epoch,
            #'model_state_dict': self.model.state_dict(), # Saves all 3 networks at once
            'model_state_dict': self.model.core.state_dict(),
            'adm_state_dict': self.model.adm.state_dict(),
            'ctp_state_dict': self.model.ctp.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': loss,
        }
        torch.save(checkpoint, os.path.join(CHECKPOINT_PATH, filename))

    def restore(self, filename):
        checkpoint_path = os.path.join(LOAD_PATH, filename)
        if not os.path.exists(checkpoint_path):
            return None
            
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.core.load_state_dict(checkpoint['model_state_dict'])
        self.model.adm.load_state_dict(checkpoint['adm_state_dict'])
        self.model.ctp.load_state_dict(checkpoint['ctp_state_dict'])
        # self.model.load_state_dict(checkpoint['model_state_dict'])
        
        if self.optimizer is None:
            self.update_stage_optimizer(stage=3)
            
        # self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint['epoch']

    def restore_seperate(self, core, adm, ctp):
        core = torch.load(core, map_location=self.device)
        self.model.core.load_state_dict(core['model_state_dict'])
        adm = torch.load(adm, map_location=self.device)
        self.model.adm.load_state_dict(adm['model_state_dict'])
        ctp = torch.load(ctp, map_location=self.device)
        self.model.ctp.load_state_dict(ctp['model_state_dict'])
        
        if self.optimizer is None:
            self.update_stage_optimizer(stage=3)
            
        # self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return core['epoch']

    # =========================================================================
    # Evaluation Suite
    # =========================================================================
    def evaluate(self, dataloader):
        self.model.core.eval()
        self.model.adm.eval()
        self.model.ctp.eval()
    
        cls_correct, cls_total = 0, 0
        adm_correct, adm_total = 0, 0
        ctp_correct, ctp_total = 0, 0
        solved, total_sat = 0, 0
    
        with torch.no_grad():
            for data_batch in dataloader:
                data_batch = data_batch.to(self.device)
                outputs, L_h, C_h, _ = self.model.core(data_batch)
                vote = torch.sigmoid(outputs)
    
                target = data_batch.y
                sat_mask = target.squeeze().bool()
    
                # --- Classification ---
                pred_binary = (vote >= 0.5).float()
                cls_correct += (pred_binary == target).sum().item()
                cls_total += data_batch.num_graphs
    
                # --- ADM (literal prediction) ---
                n_literals = data_batch.n_vars * 2 # (batch size)
                lit_votes_tensor = torch.repeat_interleave(vote, n_literals, dim=0) # (batch size * n_lits)
                adm_input = torch.cat([L_h, lit_votes_tensor[:,None]], dim=1)
                adm_logits = self.model.adm(adm_input)
                sat_mask_lit = torch.repeat_interleave(target, data_batch.n_vars * 2, dim=0).squeeze().bool()
                if sat_mask_lit.sum() > 0:
                    predicted_lit = torch.argmax(adm_logits, dim=1)
                    adm_correct += (predicted_lit[sat_mask_lit] == data_batch.lit_label.squeeze()[sat_mask_lit]).sum().item()
                    adm_total += sat_mask_lit.sum().item()
    
                # --- CTP (clause prediction) ---
                n_clauses = data_batch.n_clauses
                clause_votes_tensor = torch.repeat_interleave(vote, n_clauses, dim=0)
                ctp_input = torch.cat([C_h, clause_votes_tensor[:,None]], dim=1)
                ctp_logits = self.model.ctp(ctp_input)
                sat_mask_cls = torch.repeat_interleave(target, data_batch.n_clauses, dim=0).squeeze().bool()
                if sat_mask_cls.sum() > 0:
                    predicted_cls = torch.argmax(ctp_logits, dim=1)
                    ctp_correct += (predicted_cls[sat_mask_cls] == data_batch.clause_label.squeeze()[sat_mask_cls]).sum().item()
                    ctp_total += sat_mask_cls.sum().item()
    
                # --- Direct Solve ---
                if sat_mask.sum() > 0:
                    probs = torch.softmax(adm_logits, dim=1)[:, 1]
    
                    consistent_preds = torch.zeros_like(probs)
                    offset = 0
                    for n_v in data_batch.n_vars:
                        n_lits = n_v * 2
                        pos_half = probs[offset : offset + n_v]
                        neg_half = probs[offset + n_v : offset + n_lits]
                        var_is_true = (pos_half > neg_half).float()
                        consistent_preds[offset : offset + n_v] = var_is_true
                        consistent_preds[offset + n_v : offset + n_lits] = 1.0 - var_is_true
                        offset += n_lits
    
                    edge_index = data_batch['literal', 'to', 'clause'].edge_index
                    clause_sat = scatter(consistent_preds[edge_index[0]], edge_index[1], dim=0, dim_size=data_batch.n_clauses.sum(), reduce='max')
                    clause_batch_idx = torch.repeat_interleave(
                        torch.arange(data_batch.num_graphs, device=self.device),
                        data_batch.n_clauses
                    )
                    instance_solved = scatter(clause_sat, clause_batch_idx, dim=0, reduce='min')
                    solved += instance_solved[sat_mask].sum().item()
                    total_sat += sat_mask.sum().item()
    
        return {
            "class_acc":  cls_correct / cls_total          if cls_total  > 0 else 0,
            "adm_acc":    adm_correct / adm_total          if adm_total  > 0 else 0,
            "ctp_acc":    ctp_correct / ctp_total          if ctp_total  > 0 else 0,
            "solve_rate": solved      / total_sat          if total_sat  > 0 else 0,
        }
    
    def test(self, sat="40_SAT", unsat="40_UNSAT"):
        test_loader = testing_data_setup(sat, unsat, generate_labels=True)
        return self.evaluate(test_loader)

    def inference(self, sat="40_SAT", unsat="40_UNSAT", var_votes=False):
        dataloader = testing_data_setup(sat, unsat)
        self.model.core.eval()
        self.model.adm.eval()
        self.model.ctp.eval()
    
        all_graph_votes = []
        all_lit_embs = []
        all_clause_embs = []
        all_ctp_probs = []
        all_adm_probs = []
        all_var_votes = []

        total_samples = 0
        total_start = time.perf_counter()

        print("Extracting embeddings and running CTP predictions...")
        with torch.no_grad():
            for data_batch in dataloader:
                data_batch = data_batch.to(self.device)
                
                # --- Pass 1: NeuroSAT ---
                graph_logits, L_h, C_h, var_votes = self.model.core(data_batch)
                graph_votes = torch.sigmoid(graph_logits)

                # --- Pass 2: ADM (Fully Vectorized) ---
                n_literals = data_batch.n_vars * 2 # (batch size)
                lit_votes_tensor = torch.repeat_interleave(graph_votes, n_literals, dim=0) # (batch size * n_lits)
                adm_input = torch.cat([L_h, lit_votes_tensor[:,None]], dim=1)
                adm_logits = self.model.adm(adm_input)
                adm_probs = torch.softmax(adm_logits, dim=1)
                
                # --- Pass 3: CTP (Fully Vectorized) ---
                n_clauses = data_batch.n_clauses
                clause_votes_tensor = torch.repeat_interleave(graph_votes, n_clauses, dim=0)
                ctp_input = torch.cat([C_h, clause_votes_tensor[:,None]], dim=1)
                ctp_logits = self.model.ctp(ctp_input)
                ctp_probs = torch.softmax(ctp_logits, dim=1)
                
                # --- Store Outputs (Moving to CPU) ---                
                all_graph_votes.append(graph_votes.cpu())
                all_lit_embs.append(L_h.cpu())
                all_clause_embs.append(C_h.cpu())
                all_adm_probs.append(adm_probs.cpu())
                all_ctp_probs.append(ctp_probs.cpu())
                all_var_votes.append(var_votes.cpu())

                total_samples += data_batch.num_graphs

        total_time = (time.perf_counter() - total_start) * 1000
        latency_ms = total_time / max(1, total_samples)
        
        return {
            "graph_votes":  torch.cat(all_graph_votes),
            "lit_embs":     torch.cat(all_lit_embs),
            "clause_embs":  torch.cat(all_clause_embs),
            "adm_probs":    torch.cat(all_adm_probs),
            "ctp_probs":    torch.cat(all_ctp_probs),
            "var_votes":    torch.cat(all_var_votes),
            "latency":      latency_ms
        }

    
    # =========================================================================
    # Training 
    # =========================================================================
    def train_epoch(self, dataloader, epoch, stage=1):
        """
        Executes a single training epoch based on the specified curriculum stage.
        """
        # --- Mode Management ---
        if stage == 1:
            self.model.core.train(); self.model.adm.eval(); self.model.ctp.eval()
        elif stage == 2:
            self.model.core.eval(); self.model.adm.train(); self.model.ctp.train()
        else:
            self.model.train() # Unfreezes all sub-modules
            
        if stage == 3: 
            self.uc_loss_fn.train()
        else:
            self.uc_loss_fn.eval()
            
        total_loss = 0
        total_samples = 0 
        
        for i, batch_data in enumerate(dataloader):
            current_batch_size = batch_data.num_graphs
            batch_data = batch_data.to(self.device)

            # 1. Core Forward Pass
            outputs, L_h, C_h, _ = self.model.core(batch_data)
            vote = torch.sigmoid(outputs)
            loss_global = self.global_loss_fn(outputs, batch_data.y) 

            # --- STAGE 1: Train Core Only ---
            if stage == 1:
                self.optimizer.zero_grad()
                loss_global.backward()
                torch.nn.utils.clip_grad_norm_(self.model.core.parameters(), 0.65)
                self.optimizer.step()

                total_loss += loss_global.item() * current_batch_size
                total_samples += current_batch_size
                if (i + 1) % 10 == 0: 
                    print(f"--- Epoch {epoch + 1} | Batch {i + 1}/{len(dataloader)} | Stage 1 Loss: {loss_global.item():.6f}")
                continue

            # 2. Extract Head Inputs
            n_literals = batch_data.n_vars * 2 # (batch size)
            lit_votes_tensor = torch.repeat_interleave(vote, n_literals, dim=0) # (batch size * n_lits)
            adm_input = torch.cat([L_h, lit_votes_tensor[:,None]], dim=1)
            adm_logits = self.model.adm(adm_input)
            
            n_clauses = batch_data.n_clauses
            clause_votes_tensor = torch.repeat_interleave(vote, n_clauses, dim=0)
            ctp_input = torch.cat([C_h, clause_votes_tensor[:,None]], dim=1)
            ctp_logits = self.model.ctp(ctp_input)       
            
            # Generate Masks
            sat_mask_lit = torch.repeat_interleave(batch_data.y, n_literals, dim=0).squeeze()
            sat_mask_cls = torch.repeat_interleave(batch_data.y, n_clauses, dim=0).squeeze()
            num_sat_lits = sat_mask_lit.sum()
            num_sat_clauses = sat_mask_cls.sum()

            # --- STAGE 4: refinement Loss ---
            if stage == 4:
                raw_loss_refine, entropy_reg = compute_clause_unsat_loss(
                    adm_logits, batch_data['literal', 'to', 'clause'].edge_index, batch_data.n_vars, n_clauses)
                loss_refine = (raw_loss_refine * sat_mask_cls).sum() / (num_sat_clauses + 1e-8) + entropy_reg
        
                loss = 0.1 * loss_global + 0.9 * loss_refine
                params_to_clip = self.model.parameters()

                self.optimizer.zero_grad()
                loss.backward() 
                torch.nn.utils.clip_grad_norm_(params_to_clip, 0.65)
                self.optimizer.step()
                
                total_loss += loss.item() * current_batch_size 
                total_samples += current_batch_size
                if (i + 1) % 10 == 0: 
                    print(f"--- Epoch {epoch + 1} | Batch {i + 1}/{len(dataloader)} | Stage 4 Loss: {loss.item():.6f}")
                continue

            # --- STAGES 2, 3: Standard Supervised Loss ---
            raw_loss_lit = self.mlp_loss_fn(adm_logits, batch_data.lit_label.squeeze())
            raw_loss_cls = self.mlp_loss_fn(ctp_logits, batch_data.clause_label.squeeze())

            loss_lit = (raw_loss_lit * sat_mask_lit).sum() / (num_sat_lits + 1e-8)
            loss_clause = (raw_loss_cls * sat_mask_cls).sum() / (num_sat_clauses + 1e-8)

            if stage == 2:
                loss = (0.5 * loss_lit) + (0.5 * loss_clause)
                params_to_clip = list(self.model.adm.parameters()) + list(self.model.ctp.parameters())
            elif stage == 3:
                loss = self.uc_loss_fn([loss_global, loss_lit, loss_clause])
                params_to_clip = list(self.model.parameters()) + list(self.uc_loss_fn.parameters())        

            self.optimizer.zero_grad()
            loss.backward() 
            torch.nn.utils.clip_grad_norm_(params_to_clip, 0.65)
            self.optimizer.step()
            
            total_loss += loss.item() * current_batch_size 
            total_samples += current_batch_size

            if (i + 1) % 10 == 0: 
                print(f"--- Epoch {epoch + 1} | Batch {i + 1}/{len(dataloader)} | Stage {stage} Loss: {loss.item():.6f}")

        return total_loss / total_samples


In [266]:
# MTL training class

class MTLParadigm:
    def __init__(self, opts, train_dataset="SR_Uniform_10-40_Dataset"):
        self.opts = opts
        self.device = opts.get('device', torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        self.train_loader = training_data_setup(train_dataset, generate_labels=True)

        model = MtlNeuroSAT(
            d_model=opts.get('d_model', 64), 
            T=opts.get('T', 26)
        )
        self.trainer = MtlTrainer(model, self.device)
        self.init_random_seeds()

    def init_random_seeds(self):
        seed = self.opts.get('seed', 42)
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

    def _get_lrs(self, stage):
        """Extracts stage-specific LRs from opts, with sensible defaults."""
        if stage == 1:
            return {'core': self.opts.get('lr_stage1_core', 5e-5)}
        elif stage == 2:
            return {'adm': self.opts.get('lr_stage2_adm', 2e-5),
                    'ctp': self.opts.get('lr_stage2_ctp', 1e-4)}
        elif stage == 3:
            return {'core': self.opts.get('lr_stage3_core', 2e-5),
                    'adm':  self.opts.get('lr_stage3_adm',  4e-5),
                    'ctp':  self.opts.get('lr_stage3_ctp',  4e-5),
                    'uc':   self.opts.get('lr_stage3_uc',   4e-5)}
        elif stage == 4:
            return {'core': self.opts.get('lr_stage4', 1e-5)}

    def log_metrics(self, model_name, epoch, train_loss, val_acc, adm_acc, ctp_acc):
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        log_filename = os.path.join(LOG_PATH, f"{model_name}_training_log.txt")
        is_new_file = not os.path.exists(log_filename)
        
        with open(log_filename, 'a') as f:
            if is_new_file:
                f.write(f"--- Training Log for Model: {model_name} ---\n")
            
            log_line = (
                f"[{timestamp}] Epoch: {epoch:04d} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Acc: {val_acc:.6f} | "
                f"Adm Acc: {adm_acc:.6f} | "
                f"Ctp Acc: {ctp_acc:.6f}\n"
            )
            f.write(log_line)

    # -------------------------------------------------------------------------
    # Strategy 1: Train Everything Together
    # -------------------------------------------------------------------------
    def train_naive_together(self, test_sat, test_unsat, max_epochs=400, trial=None):
        print("Starting Naive Together Training...")
        if trial:
            model_name = f"MTL_Together_Trial{trial.number}"
        else:
            model_name = "MTL_Together"
        
        self.trainer.update_stage_optimizer(3, self._get_lrs(3))
        max_metric = 0 
        patience_counter = 0

        for epoch in range(max_epochs):
            avg_loss = self.trainer.train_epoch(self.train_loader, epoch, stage=3) 

            metrics = self.trainer.test(test_sat, test_unsat)
            c_acc, a_acc, ct_acc = metrics["class_acc"], metrics["adm_acc"], metrics["ctp_acc"]
            head_acc = (2 * a_acc * ct_acc) / (a_acc + ct_acc + 1e-8)
            tracking_metric = (head_acc + c_acc) / 2
    
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Class: {c_acc:.4f} | ADM: {a_acc:.4f} | CTP: {ct_acc:.4f}")
            self.log_metrics(model_name, epoch + 1, avg_loss, c_acc, a_acc, ct_acc)

            if tracking_metric > max_metric:
                max_metric = tracking_metric
                patience_counter = 0
                self.trainer.save(epoch, avg_loss, f"{model_name}_best.pth")
            else:
                patience_counter += 1
    
            if patience_counter >= 10 and c_acc > 0.6:
                print(">>> Early stopping")
                break

            if trial is not None:
                trial.report(tracking_metric, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        print("Together Training Finished.")
        return max_metric

    # -------------------------------------------------------------------------
    # Strategy 2: Train Sequentially (Core, then Heads)
    # -------------------------------------------------------------------------
    def train_sequential(self, test_sat, test_unsat, max_epochs=400, trial=None):
        print("Starting Sequential Training...")
        if trial:
            model_name = f"MTL_Sequential_Trial{trial.number}"
        else:
            model_name = "MTL_Sequential"
        
        # Phase 1: Core Only
        self.trainer.update_stage_optimizer(1, self._get_lrs(1))
        max_metric = 0
        patience = 0
        current_stage = 1

        for epoch in range(max_epochs):
            avg_loss = self.trainer.train_epoch(self.train_loader, epoch, stage=current_stage)
            metrics = self.trainer.test(test_sat, test_unsat)
            
            c_acc, a_acc, ct_acc = metrics["class_acc"], metrics["adm_acc"], metrics["ctp_acc"]
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Class: {c_acc:.4f} | ADM: {a_acc:.4f} | CTP: {ct_acc:.4f}")
            self.log_metrics(model_name, epoch + 1, avg_loss, c_acc, a_acc, ct_acc)

            # Phase 1 -> Phase 2 Transition
            if current_stage == 1:
                if c_acc > max_metric:
                    max_metric = c_acc
                    patience = 0
                    self.trainer.save(epoch, avg_loss, f"{model_name}_core_best.pth")
                else:
                    patience += 1

                if patience >= 10 and c_acc > 0.6:
                    print(">>> Core Matured. Freezing Core, Training Heads.")
                    current_stage = 2

                    epoch_trained = self.trainer.restore(f"{model_name}_core_best.pth")
                    self.trainer.update_stage_optimizer(2, self._get_lrs(2))
                    max_metric = 0; patience = 0

            # Phase 2 Logic
            elif current_stage == 2:
                head_acc = (2 * a_acc * ct_acc) / (a_acc + ct_acc + 1e-8)
                if head_acc > max_metric:
                    max_metric = head_acc
                    patience = 0
                    self.trainer.save(epoch, avg_loss, f"{model_name}_heads_best.pth")
                else:
                    patience += 1

                if patience >= 10 and head_acc > 0.6:
                    print(">>> Heads Matured. Early Stopping.")
                    break

                if trial is not None:
                    trial.report(head_acc, epoch)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

        print("Sequential Training Finished.")
        return max_metric

    # -------------------------------------------------------------------------
    # Strategy 3: Curriculum (1 -> 2 -> 3)
    # -------------------------------------------------------------------------
    def train_staged_curriculum(self, test_sat, test_unsat, max_epochs=400, trial=None):
        print("Starting Staged Training...")
        if trial:
            model_name = f"MTL_Staged_Trial{trial.number}"
        else:
            model_name = "MTL_Staged"
        
        # Phase 1: Core Only
        self.trainer.update_stage_optimizer(1, self._get_lrs(1))
        max_metric = 0
        patience = 0
        current_stage = 1
        buffer = 0

        for epoch in range(max_epochs):
            avg_loss = self.trainer.train_epoch(self.train_loader, epoch, stage=current_stage)
            metrics = self.trainer.test(test_sat, test_unsat)
            
            c_acc, a_acc, ct_acc = metrics["class_acc"], metrics["adm_acc"], metrics["ctp_acc"]
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Class: {c_acc:.4f} | ADM: {a_acc:.4f} | CTP: {ct_acc:.4f}")
            self.log_metrics(model_name, epoch + 1, avg_loss, c_acc, a_acc, ct_acc)

            # Phase 1 -> Phase 2 Transition
            if current_stage == 1:
                if c_acc > max_metric:
                    max_metric = c_acc
                    patience = 0
                    self.trainer.save(epoch, avg_loss, f"{model_name}_core_best.pth")
                else:
                    patience += 1

                if c_acc > 0.61:
                    print(">>> Core Reasonably Matured. Freezing Core, Training Heads.")
                    current_stage = 2
                    self.trainer.update_stage_optimizer(2, self._get_lrs(2))
                    max_metric = 0
                    patience = 0

            # Phase 2 Logic
            elif current_stage == 2:
                head_acc = (2 * a_acc * ct_acc) / (a_acc + ct_acc + 1e-8)
                buffer += 1
                if head_acc > max_metric:
                    max_metric = head_acc
                    patience = 0
                    self.trainer.save(epoch, avg_loss, f"{model_name}_heads_best.pth")
                else:
                    patience += 1

                if head_acc > 0.74 and buffer > 5:
                    print(">>> Heads Reasonably Matured. Starting Full End-to-End.")
                    current_stage = 3
                    self.trainer.update_stage_optimizer(3, self._get_lrs(3))
                    max_metric = 0
                    patience = 0

            elif current_stage == 3:
                head_acc = (2 * a_acc * ct_acc) / (a_acc + ct_acc + 1e-8)
                full_acc = (c_acc + head_acc) / 2
                if full_acc >= max_metric:
                    max_metric = full_acc
                    patience = 0
                    self.trainer.save(epoch, avg_loss, f"{model_name}_final_best.pth")
                else:
                    patience += 1

                if patience >= 15 and head_acc > 0.6:
                    print(f"Early stopping triggered at Stage 3. Training Complete.")
                    break

                if trial is not None:
                    trial.report(full_acc, epoch)
                    if trial.should_prune():
                        raise optuna.TrialPruned()
                    
        print("Staged Training Finished.")
        return max_metric


    # -------------------------------------------------------------------------
    # Strategy 4: Standalone Unsupervised Refinement (Load & Fine-Tune)
    # -------------------------------------------------------------------------
    def run_unsupervised_refinement(self, checkpoint_filename, test_sat, test_unsat, max_epochs=100):
        print(f"Loading unrefined model from '{checkpoint_filename}' for Stage 4 Refinement...")
        model_name = f"{checkpoint_filename}_Refined"

        # 1. Load the pre-trained Stage 3 model!
        epoch_trained = self.trainer.restore(checkpoint_filename)
        if epoch_trained is None:
            print("Failed to load checkpoint. Aborting refinement.")
            return

        print(f"Model loaded successfully! (Previously trained for {epoch_trained} epochs).")
        print("Switching to Unsupervised Logical Refinement...")

        # 2. Initialize Stage 4 Optimizer (Unfreeze all, very low learning rate)
        self.trainer.update_stage_optimizer(4, self._get_lrs(4))

        max_metric = 0
        patience_counter = 0

        # 3. Stage 4 Training Loop
        for epoch in range(max_epochs):
            avg_loss = self.trainer.train_epoch(self.train_loader, epoch, stage=4)            
            metrics = self.trainer.test(test_sat, test_unsat)
            
            solve_rate = metrics["solve_rate"]
            c_acc = metrics["class_acc"]
            
            print(f"Refinement Epoch {epoch+1} | Loss: {avg_loss:.4f} | Solve Rate: {solve_rate*100:.2f}% | Class Acc: {c_acc:.4f}")

            # Checkpoint based on the logical solve rate
            if solve_rate > max_metric:
                max_metric = solve_rate
                patience_counter = 0
                self.trainer.save(epoch, avg_loss, f"{model_name}_best.pth")
                print(f"  -> New best solve rate! Saved model.")
            else:
                patience_counter += 1

            if patience_counter >= 15:
                print(f"\n>>> Early stopping triggered. Refinement complete. Best Solve Rate: {max_metric*100:.2f}%")
                break
                
        print("Unsupervised Refinement Finished.")
        return max_metric
    

In [267]:
# MTL training


def mtl_training():
    opts = {
        'd_model': 64,
        'T': 26,
        'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        'seed': 42,
        'lr_stage3_core': 3.69e-5,
        'lr_stage3_adm':  2.27e-5,
        'lr_stage3_ctp':  1.13e-5
    }
    
    # Initialize the Paradigm
    experiment = MTLParadigm(opts, train_dataset="SR_Uniform_10-40_Dataset")
    
    # Run training
    experiment.train_naive_together(test_sat="40_SAT", test_unsat="40_UNSAT")
    # experiment.train_sequential(test_sat="40_SAT", test_unsat="40_UNSAT")
    # experiment.train_staged_curriculum(test_sat="40_SAT", test_unsat="40_UNSAT")


# mtl_training()

# import shutil

# shutil.make_archive('zipped_NNSAT', 'zip', '/kaggle/working/NNSAT_Project')

# from IPython.display import FileLink
# FileLink(r'zipped_NNSAT.zip')

In [268]:
# automated tuning 

import optuna
from optuna.pruners import SuccessiveHalvingPruner
import torch.optim as optim

import optuna

def optuna_objective(trial):
    opts = {
        'lr_stage1_core': trial.suggest_float('lr_stage1_core', 2e-5, 7e-5, log=True),
        'lr_stage2_adm':  trial.suggest_float('lr_stage2_adm',  1e-5, 1e-3, log=True),
        'lr_stage2_ctp':  trial.suggest_float('lr_stage2_ctp',  1e-5, 1e-3, log=True),
        #'lr_stage3_core': trial.suggest_float('lr_stage3_core', 1e-5, 5e-5, log=True),
        #'lr_stage3_adm':  trial.suggest_float('lr_stage3_adm',  1e-5, 1e-3, log=True),
        #'lr_stage3_ctp':  trial.suggest_float('lr_stage3_ctp',  1e-5, 1e-3, log=True),
        #'lr_stage3_uc':   trial.suggest_float('lr_stage3_uc',   1e-5, 1e-3, log=True),
        'd_model': 64,
        'T': 26,
        'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        'seed': 42
    }

    experiment = MTLParadigm(opts)
    metrics = experiment.train_sequential(
        test_sat="40_SAT", 
        test_unsat="40_UNSAT",
        max_epochs=400,
        trial=trial
    )
    return metrics  


def auto_hyperparameter_tuning():
    # Setup the Optuna Pruner
    pruner = SuccessiveHalvingPruner(
        min_resource=20,          # Allow trials to run for at least 20 epochs before judging them
        reduction_factor=3        # Halves the number of trials each round
    )
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(optuna_objective, n_trials=10)

    print("\n==================================")
    print("Hyperparameter Tuning Complete!")
    print("Best Trial:", study.best_trial.number)
    print("Best Accuracy:", study.best_value)
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    print("==================================")


# auto_hyperparameter_tuning()

# import shutil

# shutil.make_archive('zipped_NNSAT', 'zip', '/kaggle/working/NNSAT_Project')

# from IPython.display import FileLink
# FileLink(r'zipped_NNSAT.zip')

In [269]:
# refinement


def compute_clause_unsat_loss(adm_logits, edge_index, n_vars, n_clauses):
    probs = torch.softmax(adm_logits, dim=1)[:, 1]

    lit_offsets = torch.cat([torch.tensor([0], device=DEVICE),
                              torch.cumsum(n_vars * 2, dim=0)[:-1]])
    lit_to_graph_offset = torch.repeat_interleave(lit_offsets, n_vars * 2)
    v_counts = torch.repeat_interleave(n_vars, n_vars * 2)
    rel_idx = torch.arange(len(probs), device=DEVICE) - lit_to_graph_offset
    is_neg_node = rel_idx >= v_counts

    lit_node = edge_index[0]
    lit_true_prob = torch.where(
        is_neg_node[lit_node],
        1.0 - probs[lit_node - v_counts[lit_node]],
        probs[lit_node]
    )

    # Log-space for numerical stability
    log_unsat = scatter(torch.log(1.0 - lit_true_prob + 1e-8),
                        edge_index[1], dim=0,
                        dim_size=n_clauses.sum(), reduce='sum')
    loss_j = torch.exp(log_unsat)  # per-clause unsat probability

    # Stronger entropy reg since we don't need it to fight collapse
    entropy_reg = 0.05 * (probs * (1.0 - probs)).mean()

    return loss_j, entropy_reg


def compute_penalty_diffsat_loss(adm_logits, edge_index, n_vars, n_clauses):
    probs = torch.softmax(adm_logits, dim=1)[:, 1]  # shape: [2 * sum(n_vars)]

    # 1. Identify negative literal nodes
    lit_offsets = torch.cat([torch.tensor([0], device=DEVICE),
                              torch.cumsum(n_vars * 2, dim=0)[:-1]])
    lit_to_graph_offset = torch.repeat_interleave(lit_offsets, n_vars * 2)
    v_counts = torch.repeat_interleave(n_vars, n_vars * 2)

    rel_idx = torch.arange(len(probs), device=DEVICE) - lit_to_graph_offset
    is_neg_node = rel_idx >= v_counts  # shape: [2 * sum(n_vars)]

    # 2. Map each literal node to its corresponding positive literal's index
    # Negative literal at index k maps to k - n_vars (within its graph)
    pos_lit_idx = torch.where(is_neg_node,
                               torch.arange(len(probs), device=DEVICE) - v_counts,
                               torch.arange(len(probs), device=DEVICE))
    # pos_lit_idx[k] always points to the positive version of literal k

    # 3. Compute v_tilde per variable: v_tilde = 2*p - 1 in [-1, 1]
    # Only defined for positive literal nodes (one per variable)
    v_tilde_all = 2.0 * probs - 1.0  # shape: [2 * sum(n_vars)]

    # 4. For each edge, get s_ij and the correct v_tilde_i
    lit_node = edge_index[0]
    s_ij = torch.where(is_neg_node[lit_node], -1.0, 1.0)      # sign
    v_tilde_i = v_tilde_all[pos_lit_idx[lit_node]]             # v_tilde of the variable

    # 5. Compute clause sum: sum_i(s_ij * v_tilde_i)
    clause_sum = scatter(s_ij * v_tilde_i, edge_index[1],
                         dim=0, dim_size=n_clauses.sum(), reduce='sum')

    # 6. Clause sizes m_j
    m_j = scatter(torch.ones(lit_node.shape[0], dtype=torch.float, device=DEVICE),
                  edge_index[1], dim=0, dim_size=n_clauses.sum(), reduce='sum')

    # 7. DiffSAT loss per clause (relu so only unsatisfied clauses contribute)
    loss_j = ((clause_sum-1).pow(2) - (m_j - 1).pow(2)) / (4.0 * m_j)
    loss = torch.relu(loss_j)

    # 8. Optional entropy regularization to encourage confident assignments
    entropy_reg = 0.15 * (probs * (1.0 - probs)).mean()

    return loss, entropy_reg
    

def refine_training():
    opts = {
        'd_model': 64,
        'T': 26,
        'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        'seed': 42
    }
    
    experiment = MTLParadigm(opts, train_dataset="SR_Uniform_10-40_Dataset")
    
    # Run ONLY the refinement phase on your existing model!
    experiment.run_unsupervised_refinement(
        checkpoint_filename="MTL_Staged_stage3_best.pth",
        test_sat="40_SAT", 
        test_unsat="40_UNSAT"
    )

In [270]:
# MTL test

def run_test(checkpoint_filename, sat_data="Test_40_SAT", unsat_data="Test_40_UNSAT"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing inference on {device}...")

    # 1. Create the blank model and the trainer
    model = MtlNeuroSAT(d_model=64, T=26)
    trainer = MtlTrainer(model, device)

    # 2. Use the built-in restore method
    epoch_trained = trainer.restore(checkpoint_filename)
    
    if epoch_trained is None:
        print("Inference aborted: Checkpoint not found.")
        return

    print(f"Model successfully loaded! (Trained for {epoch_trained} epochs)")
    print(f"Evaluating on {sat_data} and {unsat_data}...\n")

    # 3. Run the unified test suite
    metrics = trainer.test(sat_data, unsat_data)

    # 4. Print the final report
    print("="*45)
    print(" FINAL MODEL INFERENCE REPORT")
    print("="*45)
    print(f"Global SAT Classification Acc: {metrics['class_acc'] * 100:>6.2f}%")
    print(f"Literal Assignment Acc (ADM):  {metrics['adm_acc'] * 100:>6.2f}%")
    print(f"Clause Tier Predictor (CTP):   {metrics['ctp_acc'] * 100:>6.2f}%")
    print("-" * 45)
    print(f"Direct Solve Rate (Zero-Shot): {metrics['solve_rate'] * 100:>6.2f}%")
    print("="*45 + "\n")

    return metrics

    
# run_test("MTL_NueroSAT_loss_best.pth")


In [271]:
# MTL inference


def mtl_inference(checkpoint_filename, test_data_sat='Test_40_SAT', test_data_unsat='Test_40_UNSAT', T_val=26):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing inference on {device}...")

    # 1. Create the blank model and the trainer
    model = MtlNeuroSAT(d_model=64, T=T_val)
    trainer = MtlTrainer(model, device)

    # 2. Use the built-in restore method
    epoch_trained = trainer.restore(checkpoint_filename)
    
    if epoch_trained is None:
        print("Inference aborted: Checkpoint not found.")
        return

    print(f"Model successfully loaded! (Trained for {epoch_trained} epochs)")
    print(f"Evaluating on {test_data_sat} and {test_data_unsat}...\n")
    test_sat_data = SATDataset(data_file=test_data_sat, is_training=False, fixed_label=1)
    test_unsat_data = SATDataset(data_file=test_data_unsat, is_training=False, fixed_label=0)
    
    merged_test_data = ConcatDataset([test_sat_data, test_unsat_data])

    # 3. Run the unified test suite
    result = trainer.inference(test_data_sat, test_data_unsat)
    print(f"Extraction complete! Latency: {result['latency']:.4f} ms/sample")

    # Get the exact number of literals and clauses for every graph in the dataset
    n_lits_per_graph = [data.n_vars.item() * 2 for data in merged_test_data]
    n_clauses_per_graph = [data.n_clauses.item() for data in merged_test_data]
    
    # Split the massive tensors instantly
    lit_embs    = result['lit_embs']
    clause_embs = result['clause_embs']
    adm_probs   = result['adm_probs']
    ctp_probs   = result['ctp_probs']
    split_lit_emb    = [e.numpy() for e in torch.split(lit_embs,    n_lits_per_graph)]
    split_clause_emb = [e.numpy() for e in torch.split(clause_embs, n_clauses_per_graph)]
    split_adm_prob   = [p.numpy() for p in torch.split(adm_probs,   n_lits_per_graph)]
    split_ctp_prob   = [p.numpy() for p in torch.split(ctp_probs,   n_clauses_per_graph)]

    
    return result["graph_votes"], split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, result["latency"]

In [272]:
# test

def count_satisfy(candidate, clauses):
    """Counts how many clauses are satisfied by a given boolean assignment."""
    sat_count = 0
    all_satisfied = True
    
    for clause in clauses:
        lits = np.array(clause)
        idxs = np.abs(lits) - 1
        
        # Check if the literal is satisfied by the current assignment
        is_satisfied = (candidate[idxs] == (lits > 0))
        
        if np.any(is_satisfied):
            sat_count += 1
        else:
            all_satisfied = False
            
    return sat_count, all_satisfied
    

def test_final_model(test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT"):
    # --- Load & split results ---
    graph_votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, latency = mtl_inference('MTL_Together_best.pth', test_data_sat, test_data_unsat)

    data_sat   = read_data(test_data_sat,   is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    solved = 0
    pred_sat = 0
    pred_class_labels = []
    true_class_labels = []
    y_probs = []

    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        vote = graph_votes[i]
        y_probs.append(vote)

        pred_class_labels.append(1 if vote >= 0.5 else 0)
        if vote >= 0.5:
            pred_sat += 1
        true_class_labels.append(is_sat_ground_truth)

        if not is_sat_ground_truth:
            continue

        # adm_probs → candidate assignment, ctp_probs → tier predictions
        candidate_prob = split_adm_prob[i]   # already numpy from mtl_inference
        prob_pos_true = candidate_prob[:n_vars, 1]
        prob_neg_true = candidate_prob[n_vars:, 1]
        candidate = (prob_pos_true > prob_neg_true).astype(int)

        count, direct_solved = count_satisfy(candidate, clauses)
        if direct_solved:
            solved += 1

        if (i + 1) % 100 == 0:
            print(f"Processed {i+1}...")

    # --- Metrics ---
    y_true  = np.array(true_class_labels)
    y_probs = np.array(y_probs)
    thresholds = [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]

    print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'False Positives (UNSATs sent to WalkSAT)'}")
    print("-" * 80)

    for t in thresholds:
        preds     = (y_probs > t).astype(int)
        precision = precision_score(y_true, preds, zero_division=0)
        recall    = recall_score(y_true, preds, zero_division=0)
        try:
            tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        except ValueError:
            fp = 0
        print(f"{t:<10.2f} | {precision:<10.4f} | {recall:<10.4f} | {fp}")

    print(f"\nDirectly Solved: {solved}/{pred_sat}")
    print(f"Latency: {latency}")
    print("\nSatisfiability Classification")
    print(classification_report(
        true_class_labels, pred_class_labels,
        target_names=['Label 0 (UNSAT)', 'Label 1 (SAT)']
    ))


# test_final_model()

In [273]:
# increasing T

import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix

def count_satisfy(candidate, clauses):
    """Checks if a given candidate assignment satisfies all clauses."""
    sat_count = 0
    all_satisfied = True
    
    for clause in clauses:
        lits = np.array(clause)
        idxs = np.abs(lits) - 1
        
        # Check if any literal in the clause evaluates to True
        is_satisfied = (candidate[idxs] == (lits > 0))
        
        if np.any(is_satisfied):
            sat_count += 1
        else:
            all_satisfied = False
            
    return sat_count, all_satisfied

def evaluate_T_sweep(checkpoint_filename, test_data_sat="Test_200_SAT", test_data_unsat="Test_200_UNSAT", T_values=[26, 50, 100, 250, 500, 1000]):
    # Load raw data once to save time (used for clause evaluation)
    data_sat   = read_data(test_data_sat,   is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat
    
    # Ground truth labels: 1 for SAT, 0 for UNSAT
    y_true = np.array([1] * len(data_sat) + [0] * len(data_unsat))

    print("=" * 80)
    print(f"STARTING T-SWEEP ABLATION ({test_data_sat} / {test_data_unsat})")
    print(f"Testing T values: {T_values}")
    print("=" * 80)

    for T in T_values:
        print(f"\n\n{'='*30} T = {T} {'='*30}")
        
        # 1. Call your new MTL Inference function
        graph_votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, latency = mtl_inference(
            checkpoint_filename=checkpoint_filename,
            test_data_sat=test_data_sat,
            test_data_unsat=test_data_unsat,
            T_val=T
        )
        
        y_probs = np.array(graph_votes)
        pred_class_labels = (y_probs >= 0.5).astype(int)
        
        solved = 0
        pred_sat_count = np.sum(pred_class_labels)

        # 2. Evaluate Assignment Validity (Only for true SAT instances)
        for i, (clauses, n_vars, is_sat) in enumerate(data):
            if not is_sat:
                continue
            
            # Get the assignment probabilities for this specific graph
            candidate_prob = split_adm_prob[i]
            
            # Assuming candidate_prob is shape (n_vars, 2), get the argmax for the binary assignment
            candidate = np.argmax(candidate_prob, axis=1)
            
            _, direct_solved = count_satisfy(candidate, clauses) 
            
            if direct_solved:
                solved += 1

        # 3. Print Core Metrics
        # Note: 'latency' is per-sample from mtl_inference. Multiplying by len(data) gets total time.
        total_inference_time = latency * len(data) / 1000  # Converted to seconds
        
        print(f"Total Inference Time: {total_inference_time:.3f} s ({latency:.4f} ms/sample)")
        print(f"Directly Solved:      {solved} / {pred_sat_count} (Predicted SAT)")
        print("-" * 80)
        
        # 4. Threshold Analysis
        thresholds = [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
        print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'False Positives (Wasted WalkSATs)'}")
        print("-" * 80)
        
        for t in thresholds:
            preds = (y_probs > t).astype(int)
            precision = precision_score(y_true, preds, zero_division=0)
            recall = recall_score(y_true, preds, zero_division=0)
            
            try:
                tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
            except ValueError:
                fp = 0 
                
            print(f"{t:<10.2f} | {precision:<10.4f} | {recall:<10.4f} | {fp}")

        print("-" * 80)


# evaluate_T_sweep('MTL_NueroSAT_loss_best.pth')

In [274]:
# hamming dist

import numpy as np
import math

def hamming_distance(solver_solution, ground_truth):
    """Calculates the normalized Hamming distance between two arrays."""
    # Vectorized for significant speedup over standard Python loops
    return np.mean(solver_solution != ground_truth)

def hammingd_experiment(checkpoint_filename, test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT"):
    # 1. Run the new MTL inference pipeline
    graph_votes, _, _, split_adm_prob, _, latency = mtl_inference(
        checkpoint_filename, test_data_sat, test_data_unsat
    )
    
    # 2. Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    # dist_bin stores [count, total_hamming_distance] for each of the 10 confidence bins (0.0 to 1.0)
    dist_bin = [[0, 0.0] for _ in range(10)] 
    
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        if not is_sat_ground_truth:
            continue

        vote = graph_votes[i]
        
        # split_adm_prob is already a list of numpy arrays from mtl_inference
        candidate_prob = split_adm_prob[i] 
        candidate = np.argmax(candidate_prob, axis=1)
        
        _, direct_solved = count_satisfy(candidate, clauses) 
        
        assignment = None
        if not direct_solved:
            # Assuming get_close_assignment is defined elsewhere in your codebase
            assignment = get_close_assignment(candidate, clauses, n_vars)
            
        if assignment is not None:
            hd = hamming_distance(candidate, assignment)
            
            # Prevent IndexError: if vote is exactly 1.0, math.floor returns 10. Cap it at 9.
            idx = min(math.floor(vote * 10), 9) 
            
            dist_bin[idx][0] += 1
            dist_bin[idx][1] += hd

    # 3. Print Results
    print("\n" + "="*50)
    print("HAMMING DISTANCE BY CONFIDENCE BINS")
    print("="*50)
    
    confidence_floor = 0.0
    for count, total_hd in dist_bin:
        confidence_ceil = round(confidence_floor + 0.1, 2)
        
        if count == 0:
            print(f"Confidence {confidence_floor:.2f}-{confidence_ceil:.2f}: mean distance 0.0000 between 0 problems.")
        else:
            mean_hd = total_hd / count
            print(f"Confidence {confidence_floor:.2f}-{confidence_ceil:.2f}: mean distance {mean_hd:.4f} between {count} problems.")
            
        confidence_floor = confidence_ceil


# hammingd_experiment("MTL_NueroSAT_loss_best.pth")

In [275]:
# guided WalkSAT

def guided_walksat(clauses, n_vars, max_flips=10000, p_noise=0.5, initial_assignment=None, var_weights=None, clause_weights=None):
    if initial_assignment is not None:
        assignment = np.array(initial_assignment, dtype=int)
    else:
        assignment = np.random.randint(2, size=n_vars)

    # Pre-process clauses for faster access
    c_vars = [[abs(l)-1 for l in c] for c in clauses] # list of lists of variable indices 
    c_pols = [[1 if l > 0 else 0 for l in c] for c in clauses] # list of lists of polarities (1 if pos, 0 if neg)
    
    # Build a map of {variable_index -> [clause_indices]}
    # This allows us to only update relevant clauses when we flip a variable
    var_to_clauses = [[] for _ in range(n_vars)]
    for i, c in enumerate(c_vars):
        for v in c:
            var_to_clauses[v].append(i)

    # Compute Initial Satisfaction
    sat_counts = np.zeros(len(clauses), dtype=int)
    unsat_stack = [] 
    for i, (lits, pols) in enumerate(zip(c_vars, c_pols)):
        satisfied_term_count = 0
        for var_idx, pol in zip(lits, pols):
            if assignment[var_idx] == pol:
                satisfied_term_count += 1
        
        sat_counts[i] = satisfied_term_count
        if satisfied_term_count == 0:
            unsat_stack.append(i)

    for step in range(max_flips):
        if len(unsat_stack) == 0:
            return assignment, step # solved
            
        rand_idx = random.randint(0, len(unsat_stack) - 1)
        target_c_idx = unsat_stack[rand_idx]
        
        candidates = c_vars[target_c_idx]

        best_break = float('inf')
        best_candidates = []
        
        for candidate in candidates:
            break_score = 0
            
            # Check all clauses this variable appears in
            for c_idx in var_to_clauses[candidate]:
                if sat_counts[c_idx] == 1:
                    # Verify if candidate is the one satisfying it
                    c_lits = c_vars[c_idx]
                    c_ps = c_pols[c_idx]
                    idx_in_c = c_lits.index(candidate)

                    if assignment[candidate] == c_ps[idx_in_c]:
                        if clause_weights is not None:
                            # higher score for breaking brittle clause
                            break_score += clause_weights[c_idx]
                        else:
                            # uniform break score
                            break_score += 1
            
            if break_score < best_break:
                best_break = break_score
                best_candidates = [candidate]
            elif break_score == best_break:
                best_candidates.append(candidate)

        # Free Move
        if best_break == 0:
            flip_var = random.choice(best_candidates)
        # Random Move
        elif random.random() < p_noise:
            # Guided Walk
            if var_weights is not None:
                flip_var = random.choices(candidates, weights=[var_weights[i] for i in candidates], k=1)[0]
            # Random Walk
            else:
                flip_var = random.choice(candidates)
        # Greedy Move
        else:            
            flip_var = random.choice(best_candidates)

        # apply flip
        old_val = assignment[flip_var]
        new_val = 1 - old_val
        assignment[flip_var] = new_val
        
        # Update SAT Counts
        # We only check clauses that contain the flipped variable
        for clause_idx in var_to_clauses[flip_var]:
            # Get the polarity of the flipped variable in this specific clause
            lits = c_vars[clause_idx]
            pols = c_pols[clause_idx]
            idx = lits.index(flip_var)
            pol = pols[idx]
            
            # If new_val matches polarity -> We just satisfied it -> count +1
            # If old_val matched polarity -> We just broke it -> count -1
            if new_val == pol:
                sat_counts[clause_idx] += 1
            else:
                sat_counts[clause_idx] -= 1
        
        # refresh UNSAT Stack
        unsat_stack = np.where(sat_counts == 0)[0]

    return None, max_flips # Failed

In [276]:
# NeuroSAT-s

def NNs_inference(test_data_sat='Test_40_SAT', test_data_unsat='Test_40_UNSAT', T_val=26):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing inference on {device}...")

    # 1. Create the blank model and the trainer
    model = SepMtlNeuroSAT(d_model=64, T=T_val)
    trainer = MtlTrainer(model, device)

    # 2. Use the built-in restore method
    epoch_trained = trainer.restore_seperate(LOAD_PATH_CORE, LOAD_PATH_ADM, LOAD_PATH_CTP)
    
    if epoch_trained is None:
        print("Inference aborted: Checkpoint not found.")
        return

    print(f"Model successfully loaded! (Trained for {epoch_trained} epochs)")
    print(f"Evaluating on {test_data_sat} and {test_data_unsat}...\n")
    test_sat_data = SATDataset(data_file=test_data_sat, is_training=False, fixed_label=1)
    test_unsat_data = SATDataset(data_file=test_data_unsat, is_training=False, fixed_label=0)
    
    merged_test_data = ConcatDataset([test_sat_data, test_unsat_data])

    # 3. Run the unified test suite
    result = trainer.inference(test_data_sat, test_data_unsat)
    print(f"Extraction complete! Latency: {result['latency']:.4f} ms/sample")

    # Get the exact number of literals and clauses for every graph in the dataset
    n_lits_per_graph = [data.n_vars.item() * 2 for data in merged_test_data]
    n_clauses_per_graph = [data.n_clauses.item() for data in merged_test_data]
    n_vars_per_graph = [data.n_vars.item() for data in merged_test_data]

    # Split the massive tensors instantly
    lit_embs    = result['lit_embs']
    clause_embs = result['clause_embs']
    adm_probs   = result['adm_probs']
    ctp_probs   = result['ctp_probs']
    var_votes   = result['var_votes']
    split_lit_emb    = [e.numpy() for e in torch.split(lit_embs,    n_lits_per_graph)]
    split_clause_emb = [e.numpy() for e in torch.split(clause_embs, n_clauses_per_graph)]
    split_adm_prob   = [p.numpy() for p in torch.split(adm_probs,   n_lits_per_graph)]
    split_ctp_prob   = [p.numpy() for p in torch.split(ctp_probs,   n_clauses_per_graph)]
    split_var_votes  = [vote.numpy() for vote in torch.split(var_votes, n_vars_per_graph)]

    return result["graph_votes"], split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, split_var_votes, result["latency"]

In [277]:
# hybrid sat search weights tuning

import numpy as np
import optuna
import numpy as np
import optuna

def calculate_clause_weights(clause_prob, weights=[8.885, 3.130, 0.305]):
    # Optimal Weights Found: {'w0': 6.442364140659167, 'w1': 2.3627843943281865, 'w2': 0.36800651526224787} [only median]
    # Optimal Weights Found: {'w0': 8.885423673735296, 'w1': 3.1296780044059127, 'w2': 0.3045233486978076} [median + mean]
    """Calculates weights for clauses based on predicted tier probabilities."""
    # Dot product of probabilities and weights
    cweights = clause_prob @ weights
    
    # Normalize
    cweights /= cweights.sum()
    return cweights

def calculate_uncertain_weights(votes, bias_factor=1.56):
    # Optimal Weights Found: {'bf': 1.5645092041327024}
    """Calculates variable weights based on prediction uncertainty."""
    
    votes = np.array(votes)
    weights = np.zeros_like(votes, dtype=float)
    
    k_pos = 1.0  
    k_neg = k_pos / bias_factor 

    # Vectorized exponential weighting based on certainty
    pos_mask = votes >= 0
    weights[pos_mask] = np.exp(-k_pos * np.abs(votes[pos_mask]))
    
    neg_mask = votes < 0
    weights[neg_mask] = np.exp(-k_neg * np.abs(votes[neg_mask]))
    
    # Normalize
    weights /= weights.sum()
    return weights

def calculate_unsat_weights(votes, bias_factor=4.65):  
    # Optimal Weights Found: {'bf': 4.6480789040525705}
    """Calculates variable weights favoring variables predicted as UNSAT."""
    
    votes = np.array(votes)
    
    # Scale votes and apply tanh
    s = np.tanh(votes / bias_factor)
    
    # Weight calculation pushing towards UNSAT variables
    weights = ((1 - s) / 2) ** 2
    
    # Normalize
    weights /= weights.sum()
    return weights


# --- Optuna Objective Functions ---
def clause_objective(trial, hard_instances):
    # Suggest weights for the three tiers
    w0 = trial.suggest_float("w0", 2.0, 10.0)  # Brittle: high range to 'freeze' them
    w1 = trial.suggest_float("w1", 1.0, 5.0)   # Medium
    w2 = trial.suggest_float("w2", 0.1, 1.0)   # Slack
    
    total_flips = []

    for clauses, n_vars, assignment, var_vote, tier_prob in hard_instances:
        cweights = calculate_clause_weights(tier_prob, weights=[w0, w1, w2])

        sol, flips = guided_walksat(
            clauses, 
            n_vars=n_vars, 
            initial_assignment=assignment,
            max_flips=10000, 
            clause_weights=cweights
        )
        
        if sol is not None:
            total_flips.append(flips)
        else:
            total_flips.append(20000) # Penalty for failing to solve

    return np.median(total_flips) + 0.2 * np.mean(total_flips)

def var_uc_objective(trial, hard_instances):
    bf = trial.suggest_float("bf", 0.0, 5.0)  
    total_flips = []

    for clauses, n_vars, assignment, var_vote, tier_prob in hard_instances:
        vweights = calculate_uncertain_weights(var_vote, bias_factor=bf)

        sol, flips = guided_walksat(
            clauses, 
            n_vars=n_vars, 
            initial_assignment=assignment,
            max_flips=10000, 
            var_weights=vweights
        )
        
        if sol is not None:
            total_flips.append(flips)
        else:
            total_flips.append(20000)

    return np.median(total_flips) + 0.2 * np.mean(total_flips)

def var_us_objective(trial, hard_instances):
    bf = trial.suggest_float("bf", 1.0, 5.0)  
    total_flips = []

    for clauses, n_vars, assignment, var_vote, tier_prob in hard_instances:
        vweights = calculate_unsat_weights(var_vote, bias_factor=bf)

        sol, flips = guided_walksat(
            clauses, 
            n_vars=n_vars, 
            initial_assignment=assignment,
            max_flips=10000, 
            var_weights=vweights
        )
        
        if sol is not None:
            total_flips.append(flips)
        else:
            total_flips.append(20000)

    return np.median(total_flips) + 0.2 * np.mean(total_flips)


# --- Main Tuning Execution ---
def weights_tuning(test_data_sat="40_SAT", test_data_unsat="40_UNSAT", tuning_type="var_us"):
    """
    Main tuning loop. 
    tuning_type options: "clause", "var_uc", "var_us"
    """
    print("Running Inference for parameter tuning...")
    
    # 1. Unpack the 7-element tuple directly from the new NNs_inference function
    graph_votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, split_var_votes, latency = NNs_inference(
        test_data_sat, test_data_unsat
    )
    
    print("Loading datasets...")
    # 2. Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    hard_instances = [] # Store: (clauses, n_vars, assignment, var_vote, tier_prob)

    print("Filtering for hard instances...")
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        # We only care about optimizing performance on true SAT problems
        if not is_sat_ground_truth:
            continue

        # Skip if the model doesn't even think it's SAT
        vote = graph_votes[i]
        if vote < 0.5:
            continue

        # 3. Pull directly from the pre-split numpy arrays returned by NNs_inference
        var_vote = split_var_votes[i]
        candidate_prob = split_adm_prob[i]
        tier_prob = split_ctp_prob[i]

        candidate = np.argmax(candidate_prob, axis=1)
        _, direct_solved = count_satisfy(candidate, clauses) 
        
        # We only want to tune on problems the NN couldn't solve directly
        if direct_solved:
            continue
       
        hard_instances.append((clauses, n_vars, candidate, var_vote, tier_prob))
      
    print(f"Found {len(hard_instances)} hard instances for tuning.")
    
    if not hard_instances:
        print("No hard instances found. Exiting tuning.")
        return

    # 4. Set up Optuna Study
    study = optuna.create_study(direction="minimize")
    
    # Select the correct objective function based on the tuning type
    if tuning_type == "clause":
        objective_func = lambda trial: clause_objective(trial, hard_instances)
    elif tuning_type == "var_uc":
        objective_func = lambda trial: var_uc_objective(trial, hard_instances)
    elif tuning_type == "var_us":
        objective_func = lambda trial: var_us_objective(trial, hard_instances)
    else:
        raise ValueError("Invalid tuning_type. Choose 'clause', 'var_uc', or 'var_us'.")

    print(f"Starting Optuna Study ({tuning_type})...")
    study.optimize(objective_func, n_trials=20)
    
    print("\n" + "="*50)
    print(f"OPTIMAL WEIGHTS FOUND ({tuning_type}):")
    print(study.best_params)
    print("="*50)

# weights_tuning(tuning_type="var_us")
# weights_tuning(tuning_type="var_uc")
# weights_tuning(tuning_type="clause")

In [278]:
# plotting

import matplotlib.pyplot as plt
import numpy as np
import os

def plot_heavy_tail(**datasets):
    plt.figure(figsize=(10, 7))
    
    for label, flips in datasets.items():
        # Sort data
        sorted_data = np.sort(flips)
        
        # Calculate CCDF: y = 1 - (index / n)
        y = 1.0 - np.arange(len(sorted_data)) / len(sorted_data)
        
        plt.loglog(sorted_data, y, marker='.', linestyle='none', alpha=0.6, label=label)
    
    plt.xlabel('Flips to Solve (Log Scale)')
    plt.ylabel('Probability of remaining unsolved (Log Scale)')
    plt.title('Tail Distribution Analysis (Log-Log CCDF)')
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.3)
    plt.tight_layout()

    filename = os.path.join(LOG_PATH, "heavy_tail_dynamic.jpg")
    plt.savefig(filename)
    plt.show()


def plot_cactus(**datasets):
    plt.figure(figsize=(12, 7))
    
    for label, flips in datasets.items():
        sorted_data = np.sort(flips)
        x_axis = np.arange(1, len(sorted_data) + 1)
        
        plt.semilogy(x_axis, sorted_data, label=label, linewidth=2)
        
        zero_shots = np.sum(sorted_data == 0)
        

    plt.xlabel('Number of Instances Solved')
    plt.ylabel('Flips Required (Log Scale)')
    plt.title('Cactus Plot: Solver Efficiency Comparison')
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.5)


    # Annotate the Zero-Shot area
    plt.axvline(x=zero_shots, color='red', linestyle='--', alpha=0.5)
    plt.text(zero_shots/2, 1, f"Zero-Shot Region\n({zero_shots} instances)", color='red', ha='center')
    
    filename = os.path.join(LOG_PATH, "cactus_plot_dynamic.jpg")
    plt.savefig(filename)
    plt.show()

In [279]:
# WalkSAT experiment

import numpy as np
import pandas as pd
import time
from scipy.stats import gmean, sem, t

def walksat_experiment(test_data_sat="Test_100_SAT", test_data_unsat="Test_100_UNSAT", num_runs=5):
    print(f"Running Inference for WalkSAT experiments (Total Runs per instance: {num_runs})...")
    
    # 1. Deterministic NN Inference (Only runs ONCE)
    votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, split_var_votes, latency = NNs_inference(
        test_data_sat, test_data_unsat
    )
    
    # 2. Load data
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    records = []
    
    print("Executing Guided WalkSAT across all instances...")
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        if not is_sat_ground_truth:
            continue

        # Extract NN heuristics for this specific instance
        vote = votes[i]
        if vote < 0.5:
            continue
        
        var_vote = split_var_votes[i]
        candidate_prob = split_adm_prob[i]
        tier_prob = split_ctp_prob[i]

        candidate = np.argmax(candidate_prob, axis=1)
        _, direct_solved = count_satisfy(candidate, clauses) 

        # Compute heuristic weights once per instance
        heuristic_configs = {
            "ws": {"args": {"initial_assignment": candidate}},
            "bc": {"args": {"initial_assignment": candidate, "clause_weights": calculate_clause_weights(tier_prob)}},
            "uc": {"args": {"initial_assignment": candidate, "var_weights": calculate_uncertain_weights(var_vote)}},
            "us": {"args": {"initial_assignment": candidate, "var_weights": calculate_unsat_weights(var_vote)}}
        }

        # --- 5x STOCHASTIC TRIAL LOOP ---
        for run_id in range(num_runs):
            instance_record = {
                'id': i,
                'run_id': run_id,
                'n_vars': n_vars,
                'nn_solved_directly': direct_solved
            }
        
            # Baseline: Pure Random Start
            start_time = time.perf_counter()
            sol, flips = guided_walksat(clauses, n_vars=n_vars, max_flips=10000)
            instance_record["rd_success"] = (sol is not None)
            instance_record["rd_flips"] = flips
            instance_record["rd_time"] = (time.perf_counter() - start_time) * 1000
        
            # Execute Neural Heuristics
            for name, config in heuristic_configs.items():
                start_time = time.perf_counter()
                
                if direct_solved:
                    flips = 0
                    success = True
                else:
                    sol, flips = guided_walksat(clauses, n_vars=n_vars, max_flips=10000, **config["args"])
                    success = (sol is not None)
                    
                # We add the NN latency to simulate end-to-end time for each independent "run"
                instance_record[f"{name}_time"] = latency + ((time.perf_counter() - start_time) * 1000)
                instance_record[f"{name}_flips"] = flips
                instance_record[f"{name}_success"] = success
        
            records.append(instance_record)

        if (i+1) % 100 == 0:
            print(f"Processed {i+1} / {len(data_sat)}...")

    # --- Statistical Analysis & Reporting ---
    df = pd.DataFrame(records)
    methods = ['rd', 'ws', 'bc', 'uc', 'us']
    
    def print_aggregated_metrics(dataframe, title, runs):
        if dataframe.empty:
            print(f"\nNo instances available for {title} analysis.")
            return

        print("\n" + "="*80)
        print(f"  {title} (Averaged over {runs} runs with 95% CI)")
        print("="*80)
        
        # Calculate stats strictly per run first, then average those stats
        run_stats = {m: {'success': [], 'median_flips': [], 'mean_flips': [], 'sgm': [], 'throughput': []} for m in methods}
        
        for r in range(runs):
            run_df = dataframe[dataframe['run_id'] == r]
            if run_df.empty: continue
            
            for m in methods:
                run_stats[m]['success'].append(run_df[f'{m}_success'].mean() * 100)
                
                succ_df = run_df[run_df[f'{m}_success']]
                if not succ_df.empty:
                    run_stats[m]['median_flips'].append(succ_df[f'{m}_flips'].median())
                    run_stats[m]['mean_flips'].append(succ_df[f'{m}_flips'].mean())
                    
                    total_time = succ_df[f'{m}_time'].sum()
                    run_stats[m]['throughput'].append((len(succ_df) * 1000) / total_time if total_time > 0 else 0)
                
                run_stats[m]['sgm'].append(gmean(run_df[f'{m}_flips'] + 10) - 10)

        # Helper to calculate Mean ± 95% Confidence Interval
        def calc_ci(data_list):
            if len(data_list) < 2: return np.mean(data_list) if data_list else 0, 0
            mean_val = np.mean(data_list)
            # t-distribution multiplier for 95% CI
            ci_val = sem(data_list) * t.ppf((1 + 0.95) / 2., len(data_list)-1)
            return mean_val, ci_val

        print("\n[1] SOLVABILITY (%)")
        for m in methods:
            mean_val, ci_val = calc_ci(run_stats[m]['success'])
            print(f"{m.upper():<5}: {mean_val:>6.1f} ± {ci_val:>4.1f}%")

        print("\n[2] SEARCH EFFICIENCY (MEDIAN & MEAN FLIPS TO SOLVE)")
        for m in methods:
            med_mean, med_ci = calc_ci(run_stats[m]['median_flips'])
            mean_mean, mean_ci = calc_ci(run_stats[m]['mean_flips'])
            print(f"{m.upper():<5}: {med_mean:>7.1f} ± {med_ci:>5.1f} median | {mean_mean:>7.1f} ± {mean_ci:>5.1f} mean")

        print("\n[3] SHIFTED GEOMETRIC MEAN (SGM-10) FLIPS")
        rd_mean, _ = calc_ci(run_stats['rd']['sgm'])
        print(f"SGM (RD): {rd_mean:.2f}")
        for m in ['ws', 'bc', 'uc', 'us']:
             m_mean, m_ci = calc_ci(run_stats[m]['sgm'])
             speedup = rd_mean / m_mean if m_mean > 0 else 0
             print(f"SGM ({m.upper()}): {m_mean:>7.2f} ± {m_ci:>5.2f}  | Speedup: {speedup:.2f}x")

        print("\n[4] THROUGHPUT (SOLVED / SECOND)")
        for m in methods:
            tp_mean, tp_ci = calc_ci(run_stats[m]['throughput'])
            print(f"{m.upper():<5}: {tp_mean:>7.2f} ± {tp_ci:>5.2f} solved/sec")

    # Execute Aggregated Printing
    print_aggregated_metrics(df, "ALL SAT INSTANCES", num_runs)
    
    needs_refinement = df[~df['nn_solved_directly']].copy()
    print_aggregated_metrics(needs_refinement, "REFINEMENT RESULTS (HARD INSTANCES ONLY)", num_runs)

    print("\n" + "="*80)

    # --- Plotting (Strictly Final Run Only) ---
    final_run_df = df[df['run_id'] == (num_runs - 1)]
    
    results_to_plot = {
        "Random Baseline": final_run_df['rd_flips'].values,
        "Warm Start": final_run_df['ws_flips'].values,
        "Brittle Clause Heuristic": final_run_df['bc_flips'].values,
        "Uncertain Variable Guidance": final_run_df['uc_flips'].values,
        "Unsatisfiable Variable Guidance": final_run_df['us_flips'].values
    }
    
    plot_cactus(**results_to_plot)
    plot_heavy_tail(**results_to_plot)

# walksat_experiment()

In [280]:
# guided Ranger

def binary_closure(knowledge_set, n_vars):
    new_units = set()
    binaries = [c for c in knowledge_set if len(c) == 2]
    
    lit_to_binaries = {l: set() for l in range(-n_vars, n_vars + 1) if l != 0}
    for c in binaries:
        for l in c:
            lit_to_binaries[l].add(c)

    # Resolve binary pairs
    for pivot in range(1, n_vars + 1):
        pos_binaries = lit_to_binaries[pivot]
        neg_binaries = lit_to_binaries[-pivot]
        
        for c1 in pos_binaries:
            for c2 in neg_binaries:
                res_set = (set(c1) - {pivot}) | (set(c2) - {-pivot})
                if any(-lit in res_set for lit in res_set):
                    continue
                    
                new_clause = tuple(sorted(list(res_set)))
                
                if len(new_clause) == 1:
                    new_units.add(new_clause[0])                    
                knowledge_set.add(new_clause)
                    
    return new_units
    

def propagate_unit(active_clauses, lit_to_clauses, new_unit, knowledge_set):
    unit_queue = [new_unit]
    
    while unit_queue:
        u_lit = unit_queue.pop(-1)
        
        clashing_indices = list(lit_to_clauses[-u_lit])
        for idx in clashing_indices:
            target_c = active_clauses[idx]

            res = tuple(l for l in target_c if l != -u_lit)
            
            if len(res) == 0:
                return True
            
            for lit in active_clauses[idx]:
                lit_to_clauses[lit].discard(idx)
            for lit in res:
                lit_to_clauses[lit].add(idx)
            active_clauses[idx] = res

            if len(res) <= 2:
                knowledge_set.add(res)

            if len(res) == 1 and res[0] not in unit_queue:
                unit_queue.append(res[0])
                
    return False


def guided_ranger(clauses, n_vars, k=200, w=20, max_flips=10000, p_i=0.1, p_t=0.9, p_g=0.5, core_prob=None, var_brittleness=None):
    # pure literals
    all_lits = set()
    for c in clauses:
        all_lits.update(c)
    pure_lits = {l for l in all_lits if -l not in all_lits}

    # clause_to_idx_in_source
    clauses = [tuple(sorted(c)) for c in clauses]
    clause_to_source = {c: i for i,c in enumerate(clauses)}

    take = min(len(clauses), k)
    if core_prob is not None:
        # top k core-prob clauses 
        sorted_indices = sorted(range(len(clauses)), key=lambda i: core_prob[i], reverse=True)
        start_indices = set([i for i in sorted_indices[:take]])
    else:
        start_indices = set(random.sample(range(len(clauses)), take))
    unchosen_indices = set(range(len(clauses))) - start_indices
    active_clauses = [clauses[i] for i in start_indices]

    knowledge_set = set() # two binary clause with flips lits are considered different. Painful to solve since using sets.

    run_stats = {"resolvable_pairs": 0, "avg_lengths": [], "min_length": w}
    
    # lit_to_clauses[lit] = {indices of active clauses containing that lit}
    lit_to_clauses = {lit: set() for lit in range(-n_vars, n_vars + 1) if lit != 0}
    for idx, clause in enumerate(active_clauses):
        for lit in clause:
            lit_to_clauses[lit].add(idx)

    for steps in range(max_flips):
        if steps % 50 == 0:
            current_lengths = [len(c) for c in active_clauses]
            run_stats["avg_lengths"].append(np.mean(current_lengths))
            run_stats["min_length"] = min(run_stats["min_length"], min(current_lengths))

        
        if steps % 250 == 0:
            u_lits = binary_closure(knowledge_set, n_vars)
            
            if u_lits:
                for u in u_lits:
                    if (u,) not in knowledge_set:
                        knowledge_set.add((u,))
                        if propagate_unit(active_clauses, lit_to_clauses, u, knowledge_set):
                            return True, steps, run_stats

        
        # refresh memory
        if steps % 50 == 0 and knowledge_set:
            idx_to_remove = random.randint(0, len(active_clauses) - 1) # may duplicate knowledge set but who cares Oo
            old_clause = active_clauses[idx_to_remove]
            new_clause = random.choice(list(knowledge_set))
            
            # Update literal map
            for lit in old_clause:
                lit_to_clauses[lit].remove(idx_to_remove)
            for lit in new_clause:
                lit_to_clauses[lit].add(idx_to_remove)
            active_clauses[idx_to_remove] = new_clause

            # refresh unchosen indices
            unchosen_indices.add(clause_to_source.get(old_clause, -1))
            unchosen_indices.discard(-1) # -1 if not from source

        # Random Move
        if random.random() < p_i:
            # eligible_indices = [i for i, c in enumerate(active_clauses) if len(c) > 2]
            # if eligible_indices:
            #     idx_to_remove = random.choice(eligible_indices)
            # else:
            idx_to_remove = random.randint(0, len(active_clauses) - 1)
            old_clause = active_clauses[idx_to_remove]

            
            if knowledge_set and random.random() < 0.3:
                new_clause = random.choice(list(knowledge_set))
                
                 # Update literal map
                for lit in old_clause:
                    lit_to_clauses[lit].remove(idx_to_remove)
                for lit in new_clause:
                    lit_to_clauses[lit].add(idx_to_remove)
                active_clauses[idx_to_remove] = new_clause
        
                # refresh unchosen indices
                unchosen_indices.add(clause_to_source.get(old_clause, -1))
                unchosen_indices.discard(-1) # -1 if not from source
                
            # Pick a random unchosen index from the original formula to swap in
            elif unchosen_indices:
                new_idx_from_source = random.choice(tuple(unchosen_indices))
                new_clause = clauses[new_idx_from_source]
                
                # Update literal map
                for lit in old_clause:
                    lit_to_clauses[lit].remove(idx_to_remove)
                for lit in new_clause:
                    lit_to_clauses[lit].add(idx_to_remove)
                active_clauses[idx_to_remove] = new_clause
        
                # refresh unchosen indices
                unchosen_indices.remove(new_idx_from_source)
                unchosen_indices.add(clause_to_source.get(old_clause, -1))
                unchosen_indices.discard(-1) # -1 if not from source
        # Greedy Move
        else:
            # guided resolve
            if var_brittleness is not None:
                pivot = random.choices(range(1, n_vars+1), var_brittleness, k=1)[0]
            # random resolve
            else:
                pivot = random.randint(1, n_vars)
                
            pos_indices = lit_to_clauses[pivot]
            neg_indices = lit_to_clauses[-pivot]
            if pos_indices and neg_indices:
                run_stats["resolvable_pairs"] += 1
                # Pick one clause from each side
                idx1 = random.choice(tuple(pos_indices))
                idx2 = random.choice(tuple(neg_indices))
                
                c1, c2 = active_clauses[idx1], active_clauses[idx2]
                
                # Perform Resolution
                res_set = (set(c1) - {pivot}) | (set(c2) - {-pivot})
                res_set = tuple(sorted(list(res_set)))

                if len(res_set) == 0:
                    return True, steps, run_stats #success

                # Tautology check: e.g., if [1, -1] is in result, discard
                if any(-lit in res_set for lit in res_set) or len(res_set) > w:
                    continue

                if len(res_set) <= 2:
                    knowledge_set.add(res_set)
            
                    if len(res_set) == 1:
                        if propagate_unit(active_clauses, lit_to_clauses, res_set[0], knowledge_set):
                            return True, steps, run_stats
                        
                        if len(c1) > len(c2):
                            target_idx = idx1 
                        else:
                            target_idx = idx2
                        old_clause = active_clauses[target_idx]
    
                        for lit in old_clause:
                            lit_to_clauses[lit].remove(target_idx)
                        for lit in res_set:
                            lit_to_clauses[lit].add(target_idx)
                        active_clauses[target_idx] = res_set
    
                        unchosen_indices.add(clause_to_source.get(old_clause, -1))
                        unchosen_indices.discard(-1)

                # Replacement logic (p_g)
                elif random.random() < p_g:
                    # Replace the longer parent
                    if len(c1) > len(c2):
                        target_idx = idx1 
                    else:
                        target_idx = idx2
                    old_clause = active_clauses[target_idx]

                    for lit in old_clause:
                        lit_to_clauses[lit].remove(target_idx)
                    for lit in res_set:
                        lit_to_clauses[lit].add(target_idx)
                    active_clauses[target_idx] = res_set

                    unchosen_indices.add(clause_to_source.get(old_clause, -1))
                    unchosen_indices.discard(-1)
                    
        # Subsumption Check 
        if random.random() < p_t:
            lit = random.randint(1, n_vars) * random.choice([1, -1])
            shares = list(lit_to_clauses[lit])
            
            if len(shares) >= 2:
                idx_a, idx_b = random.sample(shares, 2)
                c_a, c_b = set(active_clauses[idx_a]), set(active_clauses[idx_b])
                
                if c_a.issubset(c_b) and unchosen_indices:
                    new_idx = random.choice(tuple(unchosen_indices))
                    old_clause = active_clauses[idx_b]

                    for l in c_b: 
                        lit_to_clauses[l].remove(idx_b)
                    for l in clauses[new_idx]: 
                        lit_to_clauses[l].add(idx_b)
                    active_clauses[idx_b] = clauses[new_idx]

                    unchosen_indices.remove(new_idx)
                    unchosen_indices.add(clause_to_source.get(old_clause, -1))
                    unchosen_indices.discard(-1)
                elif c_b.issubset(c_a) and unchosen_indices:
                    new_idx = random.choice(tuple(unchosen_indices))
                    old_clause = active_clauses[idx_a]

                    for l in c_a: 
                        lit_to_clauses[l].remove(idx_a)
                    for l in clauses[new_idx]: 
                        lit_to_clauses[l].add(idx_a)
                    active_clauses[idx_a] = clauses[new_idx]

                    unchosen_indices.remove(new_idx)
                    unchosen_indices.add(clause_to_source.get(old_clause, -1))
                    unchosen_indices.discard(-1)
   
        # pure literal elimination
        if random.random() < p_t:
            check_idx = random.randint(0, len(active_clauses) - 1)
            c = active_clauses[check_idx]
            
            if any(lit in pure_lits for lit in c):
                new_idx = random.choice(tuple(unchosen_indices))

                for l in c: 
                    lit_to_clauses[l].remove(check_idx)
                for l in clauses[new_idx]: 
                    lit_to_clauses[l].add(check_idx)
                active_clauses[check_idx] = clauses[new_idx]

                unchosen_indices.add(clause_to_source.get(c, -1))
                unchosen_indices.discard(-1)

    return False, max_flips, run_stats # Failed

In [281]:
# ranger experiment

import numpy as np
import pandas as pd
import time
from scipy.stats import gmean


def calculate_softmax(x):
    """Computes a numerically stable softmax over a 1D numpy array."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def get_var_brittleness(embeddings, n_vars):
    """Calculates variable brittleness using the distance to the embedding centroid."""
    center = embeddings.mean(axis=0)
    lit_distances = np.linalg.norm(embeddings - center, axis=1)
    
    # Vectorized computation instead of a for-loop: 
    stress = (lit_distances[:n_vars] + lit_distances[n_vars:2*n_vars]) / 2.0
    
    weights = calculate_softmax(stress)
    return weights

def get_core_prob(embeddings, clauses):
    """Calculates clause core probability using the distance to the embedding centroid."""
    center = embeddings.mean(axis=0)
    clause_distances = np.linalg.norm(embeddings - center, axis=1)
    
    weights = calculate_softmax(clause_distances)
    return weights

def ranger_test(test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT"):
    print("Running Inference for Ranger tests...")
    # 1. Use the new NNs_inference tuple unpacking
    graph_votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, split_var_votes, latency = NNs_inference(
        test_data_sat, test_data_unsat
    )
    
    # 2. Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    pairs = []
    cl = []
    minn = []
    solved = 0
    
    print("Executing Guided Ranger across UNSAT instances...")
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        # We only evaluate ground-truth UNSAT instances
        if is_sat_ground_truth:
            continue

        # Skip if the model falsely predicted SAT
        vote = graph_votes[i]
        if vote > 0.5:
            continue

        # Pull embeddings directly from the pre-split numpy arrays
        L_h = split_lit_emb[i]
        C_h = split_clause_emb[i]

        var_brittleness = get_var_brittleness(L_h, n_vars)
        core_prob = get_core_prob(C_h, clauses)

        # 3. Use the guided_ranger function
        res, steps, stats = guided_ranger(clauses, n_vars, var_brittleness=var_brittleness, core_prob=core_prob)
        
        if res:
            solved += 1
            
        pairs.append(stats["resolvable_pairs"])
        cl.append(np.mean(stats["avg_lengths"]))
        minn.append(stats["min_length"])

        if (i+1) % 100 == 0:
            print(f"Processed {i+1} / {len(data)}...")

    # --- Print Summary Metrics ---
    print("\n" + "="*60)
    print("RANGER TEST RESULTS (UNSAT INSTANCES)")
    print("="*60)
    
    if len(pairs) > 0:
        print(f"Instances evaluated:                   {len(pairs)}")
        print(f"Total solved:                          {solved}")
        print(f"Mean number of resolvable pairs:       {np.mean(pairs):.2f}")
        print(f"Mean average length of active clauses: {np.mean(cl):.2f}")
        print(f"Mean minimum length of active clauses: {np.mean(minn):.2f}")
    else:
        print("No instances evaluated (all were SAT or falsely predicted as SAT).")


def ranger_experiment(test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT"):
    print("Running Inference for Ranger experiments...")
    
    # 1. Use the new NNs_inference tuple unpacking
    graph_votes, split_lit_emb, split_clause_emb, split_adm_prob, split_ctp_prob, split_var_votes, latency = NNs_inference(
        test_data_sat, test_data_unsat
    )
    
    # 2. Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    records = []
    count = 0
    
    print("Executing Guided Ranger across UNSAT instances...")
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        # Evaluate only on true UNSAT instances
        if is_sat_ground_truth:
            continue

        # Skip if model predicted SAT
        vote = graph_votes[i]
        if vote > 0.5:
            continue

        # Extract pre-split numpy arrays
        L_h = split_lit_emb[i]
        C_h = split_clause_emb[i]

        var_brittleness = get_var_brittleness(L_h, n_vars)
        core_prob = get_core_prob(C_h, clauses)

        # Helper function to run and time the guided_ranger
        def run_ranger_trial(config_kwargs):
            start = time.perf_counter()
            success, steps, stats = guided_ranger(clauses, n_vars, **config_kwargs)
            search_time = (time.perf_counter() - start) * 1000
            return success, steps, stats, search_time

        # --- Execute Heuristics ---
        
        # 1. Complete (Both Heuristics)
        complete_success, complete_steps, complete_stats, complete_search_time = run_ranger_trial({
            'var_brittleness': var_brittleness, 'core_prob': core_prob
        })
        
        # 2. Core-Only
        core_success, core_steps, core_stats, core_search_time = run_ranger_trial({
            'core_prob': core_prob
        })
        
        # 3. Var-Only
        var_success, var_steps, var_stats, var_search_time = run_ranger_trial({
            'var_brittleness': var_brittleness
        })
        
        # 4. Random Baseline
        random_success, random_steps, random_stats, random_search_time = run_ranger_trial({})

        # Print a sample to monitor progress/differences
        if complete_steps != 10000 and count < 10:
            print(f"[{i}] Steps -> Complete: {complete_steps}, Core: {core_steps}, Var: {var_steps}, Random: {random_steps}")
            count += 1
            
        # --- Store Data ---
        records.append({
            'id': i,
            'n_vars': n_vars,

            'complete_time': latency + complete_search_time,
            'complete_success': complete_success,
            'complete_steps': complete_steps,
            'complete_active_clause_length': np.mean(complete_stats['avg_lengths']) if complete_stats['avg_lengths'] else 0,

            'core_time': latency + core_search_time,
            'core_success': core_success,
            'core_steps': core_steps,
            'core_active_clause_length': np.mean(core_stats['avg_lengths']) if core_stats['avg_lengths'] else 0,

            'var_time': latency + var_search_time,
            'var_success': var_success,
            'var_steps': var_steps,
            'var_active_clause_length': np.mean(var_stats['avg_lengths']) if var_stats['avg_lengths'] else 0,

            'random_time': random_search_time, # Random has 0 latency
            'random_success': random_success,
            'random_steps': random_steps,
            'random_active_clause_length': np.mean(random_stats['avg_lengths']) if random_stats['avg_lengths'] else 0
        })

        if (i+1) % 100 == 0:
            print(f"Processed {i+1} / {len(data)}...")

    # --- Analysis & Reporting ---
    df = pd.DataFrame(records)
    
    if df.empty:
        print("No instances to analyze.")
        return

    solved_by_all = df[df['complete_success'] & df['random_success'] & df['core_success'] & df['var_success']]
    total_unfiltered = len(df)
    total_all_solved = len(solved_by_all)
    
    print("\n" + "="*60)
    print("      ABLATION STUDY: GNN HEURISTICS VS RANDOM BASELINE")
    print("="*60)
    print(f"Total UNSAT Instances Tested: {total_unfiltered}")
    print(f"Instances Solved by All Methods: {total_all_solved}")
    
    methods = ['random', 'core', 'var', 'complete']

    print("\n[1] SOLVABILITY (SUCCEEDED / TOTAL)")
    for m in methods:
        success_rate = df[f'{m}_success'].mean() * 100
        total_solved = df[f'{m}_success'].sum()
        print(f"{m.upper():<10}: {success_rate:>5.1f}% ({total_solved}/{total_unfiltered})")

    print("\n[2] SEARCH EFFICIENCY (MEDIAN STEPS TO SOLVE)")
    for m in methods:
        success_df = df[df[f'{m}_success']]
        if not success_df.empty:
            median_steps = success_df[f'{m}_steps'].median()
            print(f"{m.upper():<10}: {median_steps:>7.1f} steps")
        else:
            print(f"{m.upper():<10}: No successful solves")

    print("\n[3] SHIFTED GEOMETRIC MEAN (SGM-10) SPEEDUP")
    shift = 10
    sgm = {}
    for m in methods:
        sgm[m] = gmean(df[f'{m}_steps'] + shift) - shift
    
    print(f"SGM (Random):   {sgm['random']:.2f}")
    for m in ['core', 'var', 'complete']:
        speedup = sgm['random'] / sgm[m] if sgm[m] > 0 else 0
        print(f"SGM ({m.capitalize() + '):':<9} {sgm[m]:>7.2f}  | Speedup: {speedup:.2f}x")

    print("\n[4] WALL-CLOCK PERFORMANCE (SEARCH + INFERENCE)")
    for m in methods:
        success_df = df[df[f'{m}_success']]
        if not success_df.empty:
            avg_time = success_df[f'{m}_time'].mean()
            total_time = success_df[f'{m}_time'].sum()
            throughput = (len(success_df) * 1000) / total_time if total_time > 0 else 0
            print(f"{m.upper():<10}: {avg_time:>6.2f} ms/sample | Throughput: {throughput:>6.2f} solved/sec")
        else:
             print(f"{m.upper():<10}: No successful solves")

    print("\n[5] LOGICAL DENSITY (MEAN ACTIVE CLAUSE LENGTH)")
    for m in methods:
        avg_len = df[f'{m}_active_clause_length'].mean()
        print(f"{m.upper():<10}: {avg_len:.3f} literals")

    print("\n" + "="*60)

    # --- Plotting ---
    results_to_plot = {
        "Random Baseline": df['random_steps'].values,
        "Unsatisfiable Core Start": df['core_steps'].values,
        "Weighted Variable Resolution": df['var_steps'].values,
        "Both Heuristics": df['complete_steps'].values,        
    }
    
    # plot_cactus(**results_to_plot)
    # plot_heavy_tail(**results_to_plot)


# ranger_test()
# ranger_experiment()

In [282]:
# ic solver

import numpy as np
import time
from sklearn.metrics import classification_report
import torch
import random

def solver_pipeline(checkpoint_filename, test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT", mflips=280):
    print("Running Inference for Solver Pipeline...")
    
    # 1. Single call to your updated NN_inference
    if checkpoint_filename == "NNs":
        graph_votes, _, _, split_adm_prob, _, _, latency = NNs_inference(
            test_data_sat, test_data_unsat, T_val=500
        )
    else:
        graph_votes, _, _, split_adm_prob, _, latency = mtl_inference(
            checkpoint_filename, test_data_sat, test_data_unsat, T_val=500
        )

    # 2. Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    true_labels = []
    pred_labels = []

    walksat_count = 0
    rescued_count = 0
    reject_count = 0
    avg_flips = []

    # 3. Only time the actual WalkSAT search loop, since NN time is tracked via `latency`
    start_search_time = time.perf_counter()

    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        true_labels.append(1 if is_sat_ground_truth else 0)

        vote = graph_votes[i]
        candidate_prob = split_adm_prob[i]

        candidate = np.argmax(candidate_prob, axis=1)
        _, direct_solved = count_satisfy(candidate, clauses) 
        
        if direct_solved:
            pred_labels.append(1)
            
        elif vote > 0.5:
            walksat_count += 1
            
            # Use guided_walksat instead of the old walksat
            sol, flips = guided_walksat(clauses, n_vars=n_vars, max_flips=mflips, initial_assignment=candidate)
            avg_flips.append(flips)
            
            if sol is not None:
                pred_labels.append(1)
                rescued_count += 1
            else:
                pred_labels.append(0)
                if not is_sat_ground_truth:
                    reject_count += 1
        else:
            pred_labels.append(0)
            
    search_time = time.perf_counter() - start_search_time
    
    # Calculate exact total times
    inference_time_s = (latency * len(data)) / 1000
    total_time_s = inference_time_s + search_time

    # --- Print Summary Metrics ---
    target_names = ['Label 0 (Unsatisfiable)', 'Label 1 (Satisfiable)']
    print("\n" + "="*60)
    print(f"SOLVER PIPELINE RESULTS (Max Flips: {mflips})")
    print("="*60)
    
    print(classification_report(true_labels, pred_labels, target_names=target_names))
    print("-" * 60)
    
    print(f"Total time for {len(data)} instances:   {total_time_s:.3f} s")
    print(f"Avg latency per instance:         {(total_time_s * 1000) / len(data):.2f} ms")
    print(f"Total pure NN inference time:     {inference_time_s:.3f} s")
    print("-" * 60)
    
    print(f"Total instances solved:                 {np.sum(pred_labels)}")
    print(f"No. of Guided WalkSAT attempted:        {walksat_count}")
    print(f"No. of rescued Satisfiable instances:   {rescued_count}")
    print(f"No. of rejected Unsatisfiable instances:{reject_count}")
    
    mean_flips = np.mean(avg_flips) if avg_flips else 0
    print(f"Mean number of WalkSAT flips:           {mean_flips:.2f}")

    return np.sum(pred_labels), mean_flips, search_time
    

def weights_tuning(checkpoint_filename, test_sat="40_SAT", test_unsat="40_UNSAT"):
    # Warmup run to initialize CUDA and caches
    print("Performing warmup run...")
    _, _, _ = solver_pipeline(checkpoint_filename, test_data_sat=test_sat, test_data_unsat=test_unsat, mflips=10)

    seeds = [25, 50, 75]
    flip_range = list(range(50, 501, 50))
    
    all_solved = []
    all_flips = []
    all_times = [] # Renamed from 'time' to avoid shadowing the built-in time module
    
    print("\n" + "="*50)
    print("STARTING MAX FLIPS TUNING")
    print("="*50)

    for seed in seeds:
        print(f"\n--- Running Seed: {seed} ---")
        
        # Enforce reproducibility
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        
        seed_solved = []
        seed_flips = []
        seed_times = []
        
        for mflips in flip_range:
            print(f"Testing Max Flips: {mflips}")
            s, f, t = solver_pipeline(checkpoint_filename, test_data_sat=test_sat, test_data_unsat=test_unsat, mflips=mflips)
            
            seed_solved.append(s)
            seed_flips.append(f)
            seed_times.append(t)

        all_solved.append(seed_solved)
        all_flips.append(seed_flips)
        all_times.append(seed_times)

    # Convert to numpy arrays of shape (len(seeds), len(flip_range))
    solved_arr = np.array(all_solved)
    flips_arr = np.array(all_flips)
    time_arr = np.array(all_times)

    # Calculate statistics across the different seeds (axis=0)
    solved_mean = np.mean(solved_arr, axis=0)
    solved_std = np.std(solved_arr, axis=0)
    
    flips_mean = np.mean(flips_arr, axis=0)
    flips_std = np.std(flips_arr, axis=0)
    
    time_mean = np.mean(time_arr, axis=0)
    time_std = np.std(time_arr, axis=0)

    # --- Print Summary ---
    print("\n" + "="*50)
    print("TUNING RESULTS SUMMARY")
    print("="*50)
    print(f"Tested Flip Ranges: {flip_range}\n")
    
    print(f"Solved Mean: {np.round(solved_mean, 2)}")
    print(f"Solved Std:  {np.round(solved_std, 2)}\n")
    
    print(f"Flips Mean:  {np.round(flips_mean, 2)}")
    print(f"Flips Std:   {np.round(flips_std, 2)}\n")
    
    print(f"Time Mean:   {np.round(time_mean, 3)}")
    print(f"Time Std:    {np.round(time_std, 3)}")
    
    # Return a dictionary so you can easily plot these later
    return {
        "flip_range": flip_range,
        "solved_mean": solved_mean, "solved_std": solved_std,
        "flips_mean": flips_mean, "flips_std": flips_std,
        "time_mean": time_mean, "time_std": time_std
    }


# tuning_stats = weights_tuning("40_SAT", "40_UNSAT")
# solver_pipeline('NNs', mflips=250)
LOAD_PATH = '/kaggle/input/datasets/weihengwong/perfect-refine/'
solver_pipeline('MTL_NueroSAT_loss_best.pth', test_data_sat="flat30-60", test_data_unsat="", mflips=375)

LOAD_PATH = '/kaggle/input/datasets/weihengwong/ucloss-refine'
solver_pipeline('MTL_NueroSAT_loss_best.pth', test_data_sat="flat30-60", test_data_unsat="", mflips=375)

solver_pipeline('NNs', test_data_sat="flat30-60", test_data_unsat="", mflips=375)


Running Inference for Solver Pipeline...
Initializing inference on cuda...
Model successfully loaded! (Trained for 4 epochs)
Evaluating on flat30-60 and ...

Cache file not found. Reading and converting raw data...
Saving processed data to ./NNSAT_Project/Cache/cached_test_1_flat30-60.pt...
Loaded 100 samples total.
Cache file not found. Reading and converting raw data...
Saving processed data to ./NNSAT_Project/Cache/cached_test_0_.pt...
Loaded 0 samples total.
Loading cached data from ./NNSAT_Project/Cache/cached_test_1_flat30-60.pt...
Loaded 100 samples total.
Loading cached data from ./NNSAT_Project/Cache/cached_test_0_.pt...
Loaded 0 samples total.
Extracting embeddings and running CTP predictions...
Extraction complete! Latency: 24.3707 ms/sample

SOLVER PIPELINE RESULTS (Max Flips: 375)
                         precision    recall  f1-score   support

Label 0 (Unsatisfiable)       0.00      0.00      0.00         0
  Label 1 (Satisfiable)       1.00      0.87      0.93       100

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Extraction complete! Latency: 23.8437 ms/sample

SOLVER PIPELINE RESULTS (Max Flips: 375)
                         precision    recall  f1-score   support

Label 0 (Unsatisfiable)       0.00      0.00      0.00       0.0
  Label 1 (Satisfiable)       0.00      0.00      0.00     100.0

               accuracy                           0.00     100.0
              macro avg       0.00      0.00      0.00     100.0
           weighted avg       0.00      0.00      0.00     100.0

------------------------------------------------------------
Total time for 100 instances:   2.658 s
Avg latency per instance:         26.58 ms
Total pure NN inference time:     2.384 s
------------------------------------------------------------
Total instances solved:                 0
No. of Guided WalkSAT attempted:        0
No. of rescued Satisfiable instances:   0
No. of rejected Unsatisfiable instances:0
Mean number of WalkSAT flips:           0.00
Running Inference for Solver Pipeline...
Initializing inf

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

Extraction complete! Latency: 23.8772 ms/sample

SOLVER PIPELINE RESULTS (Max Flips: 375)
                         precision    recall  f1-score   support

Label 0 (Unsatisfiable)       0.00      0.00      0.00       0.0
  Label 1 (Satisfiable)       0.00      0.00      0.00     100.0

               accuracy                           0.00     100.0
              macro avg       0.00      0.00      0.00     100.0
           weighted avg       0.00      0.00      0.00     100.0

------------------------------------------------------------
Total time for 100 instances:   2.663 s
Avg latency per instance:         26.63 ms
Total pure NN inference time:     2.388 s
------------------------------------------------------------
Total instances solved:                 0
No. of Guided WalkSAT attempted:        0
No. of rescued Satisfiable instances:   0
No. of rejected Unsatisfiable instances:0
Mean number of WalkSAT flips:           0.00


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

(np.int64(0), 0, 0.2756894709998505)

In [283]:
# cadical

import time
from sklearn.metrics import classification_report


def is_satisfiable(clause):
    with Cadical153(bootstrap_with=clause) as solver:
        return solver.solve()


def benchmark(test_data_sat="Test_100_SAT", test_data_unsat="Test_100_UNSAT"):
    print(f"Loading datasets for Cadical benchmark...")
    # Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    if not data:
        print("No data loaded. Exiting benchmark.")
        return

    true_labels = []
    pred_labels = []

    print(f"Running Cadical solver on {len(data)} instances...")
    start_time = time.perf_counter()
    
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        true_labels.append(1 if is_sat_ground_truth else 0)
        
        # Streamlined the append logic into a single line
        pred_labels.append(1 if is_satisfiable(clauses) else 0)

        # Added progress tracking, as Cadical can occasionally hang on hard problems
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1} / {len(data)}...")

    total_time = time.perf_counter() - start_time

    # --- Print Summary Metrics ---
    target_names = ['Label 0 (Unsatisfiable)', 'Label 1 (Satisfiable)']
    
    print("\n" + "="*60)
    print("CADICAL BENCHMARK RESULTS")
    print("="*60)
    print(classification_report(true_labels, pred_labels, target_names=target_names))
    print("-" * 60)
    print(f"Total time for {len(data)} instances:   {total_time:.3f} s")
    print(f"Avg latency per instance:         {(total_time * 1000) / len(data):.2f} ms")
    print("="*60)
# benchmark("Test_200_SAT", "Test_200_UNSAT")

In [284]:
#cc_solver


import numpy as np
import time
from sklearn.metrics import classification_report

def sat_filter_simulated_parallel(test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT", mflips=100):
    # ==========================================
    # 1. BATCH INFERENCE & DATA LOADING
    # ==========================================
    print("Running initial NN Inference...")
    
    # Use the new NNs_inference function
    graph_votes, _, _, split_adm_prob, _, latency = mtl_inference(
        'MTL_NueroSAT_loss_best.pth', test_data_sat, test_data_unsat, T_val=45
    )
    
    # Use the updated read_data function
    data_sat = read_data(test_data_sat, is_training=False, fixed_label=1)
    data_unsat = read_data(test_data_unsat, is_training=False, fixed_label=0)
    data = data_sat + data_unsat

    # The latency variable from NNs_inference is per-sample in milliseconds. 
    # Convert to seconds for the systems calculations.
    avg_inference_time = latency / 1000.0  
    total_inference_time = avg_inference_time * len(data)

    # ==========================================
    # 2. ACCUMULATORS & COUNTERS
    # ==========================================
    true_labels = []
    pred_labels = []

    walksat_count = 0
    rescued_count = 0
    direct_count = 0
    avg_flips = []

    pure_cadical_time = 0.0
    
    # Sequential starts with the cost of running inference on the whole dataset
    sequential_pipeline_time = total_inference_time 
    
    # Parallel starts at 0, time is accumulated instance-by-instance in the race
    simulated_parallel_time = 0.0

    # ==========================================
    # 3. EVALUATION LOOP
    # ==========================================
    print("Simulating Systems Parallelization...")
    for i, (clauses, n_vars, is_sat_ground_truth) in enumerate(data):
        true_labels.append(1 if is_sat_ground_truth else 0)

        vote = graph_votes[i]
        candidate_prob = split_adm_prob[i]
        candidate = np.argmax(candidate_prob, axis=1)
        
        _, direct_solved = count_satisfy(candidate, clauses) 

        # ------------------------------------------
        # TIME THREAD A: PURE CADICAL
        # ------------------------------------------
        t0_cad = time.perf_counter()
        cadical_result = is_satisfiable(clauses)  
        t_cadical = time.perf_counter() - t0_cad
        pure_cadical_time += t_cadical

        # ------------------------------------------
        # TIME THREAD B: HYBRID WALKSAT
        # ------------------------------------------
        t_walksat = 0.0
        walksat_solved = False

        if direct_solved:
            walksat_solved = True
            direct_count += 1
            pred_labels.append(1)
            
        elif vote > 0.5:
            walksat_count += 1
            
            t0_ws = time.perf_counter()
            sol, flips = guided_walksat(clauses, n_vars=n_vars, max_flips=mflips, initial_assignment=candidate)
            t_walksat = time.perf_counter() - t0_ws
            
            avg_flips.append(flips)
            
            if sol is not None:
                walksat_solved = True
                rescued_count += 1
                pred_labels.append(1)
        
        # Fallback accuracy check if NN + WalkSAT failed to prove SAT
        if not walksat_solved:
            pred_labels.append(1 if cadical_result else 0)

        # ------------------------------------------
        # CALCULATE SYSTEMS METRICS
        # ------------------------------------------
        # Sequential: We pay for WalkSAT attempt. If it failed, we ALSO pay for CaDiCaL.
        sequential_pipeline_time += t_walksat
        if not walksat_solved:
            sequential_pipeline_time += t_cadical

        # Parallel (VBS): CaDiCaL and (NN+WalkSAT) race each other.
        if walksat_solved:
            # Rescued! Time taken is whichever thread finished first.
            simulated_parallel_time += min(t_cadical, avg_inference_time + t_walksat)
        else:
            # WalkSAT failed quietly. CaDiCaL finishes at its normal time.
            # We assume parallel threads launch simultaneously, so we only wait for CaDiCaL.
            simulated_parallel_time += t_cadical

    # ==========================================
    # 4. PRINT RESULTS
    # ==========================================
    target_names = ['Label 0 (Unsatisfiable)', 'Label 1 (Satisfiable)']
    print(classification_report(true_labels, pred_labels, target_names=target_names))
    
    print("\n" + "="*50)
    print("      SYSTEMS TIMING RESULTS")
    print("="*50)
    print(f"Total NN Inference Time:  {total_inference_time:.3f} s")
    print("-" * 50)
    print(f"1. Pure CaDiCaL Time:     {pure_cadical_time:.3f} s")
    print(f"2. Sequential Pipeline:   {sequential_pipeline_time:.3f} s")
    print(f"3. Simulated Parallel:    {simulated_parallel_time:.3f} s  <-- (VBS Limit)")
    print("="*50)
    
    print(f"Total instances evaluated:     {len(data)}")
    print(f"Total instances correctly classified: {np.sum(np.array(true_labels) == np.array(pred_labels))}")
    print(f"Direct Solved by NN:           {direct_count}")
    print(f"WalkSAT attempted:             {walksat_count}")
    print(f"WalkSAT successfully rescued:  {rescued_count}")
    
    mean_flips = np.mean(avg_flips) if avg_flips else 0
    print(f"Mean WalkSAT flips (when attempted): {mean_flips:.1f}")
    
    return total_inference_time, pure_cadical_time, sequential_pipeline_time, simulated_parallel_time, direct_count, walksat_count, rescued_count, mean_flips


def restart_filter_eval(test_data_sat="Test_40_SAT", test_data_unsat="Test_40_UNSAT", mflips=100, num_runs=5):
    nn_inf = []
    cadical = []
    sequential = []
    parallel = []
    dc = []
    wa = []
    wr = []
    fl = []
    
    print(f"Starting {num_runs} evaluation runs. This may take a few minutes...")
    
    for i in range(num_runs):
        print(f"\n--- Run {i+1}/{num_runs} ---")
        nn_t, pct, seq_t, par_t, d_count, w_count, r_count, m_flips_val = sat_filter_simulated_parallel(test_data_sat, test_data_unsat, mflips)
        
        nn_inf.append(nn_t)
        cadical.append(pct)
        sequential.append(seq_t)
        parallel.append(par_t)
        dc.append(d_count)
        wa.append(w_count)
        wr.append(r_count)
        fl.append(m_flips_val)

    # --- Print Aggregated Results ---
    print("\n" + "="*55)
    print(f"      AGGREGATED SYSTEMS TIMING RESULTS (N={num_runs})")
    print("="*55)
    print(f"0. NN Inference Time:     {np.mean(nn_inf):.3f} ± {np.std(nn_inf):.3f} s")
    print(f"1. Pure CaDiCaL Time:     {np.mean(cadical):.3f} ± {np.std(cadical):.3f} s")
    print(f"2. Sequential Pipeline:   {np.mean(sequential):.3f} ± {np.std(sequential):.3f} s")
    print(f"3. Simulated Parallel:    {np.mean(parallel):.3f} ± {np.std(parallel):.3f} s")
    print("-" * 55)
    print("            AGGREGATED SEARCH STATISTICS")
    print("-" * 55)
    print(f"Mean Direct solved:             {np.mean(dc):.1f} ± {np.std(dc):.1f}")
    print(f"Mean WalkSAT attempted:         {np.mean(wa):.1f} ± {np.std(wa):.1f}")
    print(f"Mean WalkSAT rescues:           {np.mean(wr):.1f} ± {np.std(wr):.1f}")
    print(f"Mean WalkSAT flips (when used): {np.mean(fl):.1f} ± {np.std(fl):.1f}")
    print("="*55)

# Run the evaluation
# restart_filter_eval("uf200", "uuf200", mflips=560, num_runs=5)